# 성남시 젠트리피케이션 위험도 분석 시스템 구축 및 상생 체계 제안

## 최종 통합 분석용 Jupyter Notebook

### 작성 메타
- 작성자: 팀 통합 정리
- 작성일: 2026-04-29
- 노트북명: 성남시_젠트리피케이션_최종통합분석
- 작성 목적: 은비, 근수, 지륜, 성주 개인 통합 정리 내용을 템플릿 형식에 맞춰 하나의 노트북으로 취합한다.
- 원본 노트북: 수정하지 않고 읽기 전용으로 참조함.


## 1. 이 노트북에서 할 일

이 노트북에서 아래 작업을 순서대로 수행합니다. 완료한 항목은 체크하면서 진행하세요.

- [ ] 개인별 clean 데이터 불러오기
- [ ] 공통 키 점검
- [ ] master table 생성
- [ ] EDA
- [ ] 통계 검정
- [ ] 머신러닝 모델 2개 적용
- [ ] 모델 비교
- [ ] 위험도 점수 생성
- [ ] 대시보드용 파일 저장
- [ ] 공모전 제출용 요약 작성

In [ ]:
# ==================================================
# 2. 환경 설정
# ==================================================
# 초보자 안내:
# - 이 셀은 노트북 전체에서 공통으로 사용할 라이브러리, 폴더 경로, 파일명을 한 번에 정리하는 곳입니다.
# - 실제 파일명이 다르면 아래 FILE 변수만 수정하면 됩니다.
# - 이후 셀에서는 여기서 정의한 경로와 변수명을 그대로 사용합니다.

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, classification_report, mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

PROJECT_ROOT = Path('.')
INPUT_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output'
FIG_DIR = OUTPUT_DIR / 'figures'

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

GEUNSU_STORE_FILE = INPUT_DIR / 'geunsu_store_clean.csv'
GEUNSU_SALES_FILE = INPUT_DIR / 'geunsu_sales_clean.csv'
GEUNSU_LANDPRICE_FILE = INPUT_DIR / 'geunsu_landprice_clean.csv'

JIRYUN_T13_FILE = INPUT_DIR / 't13_seongnam_final.parquet'
JIRYUN_T25_FILE = INPUT_DIR / 't25_seongnam_final.parquet'
JIRYUN_T26_FILE = INPUT_DIR / 't26_seongnam_final.parquet'
JIRYUN_T27_FILE = INPUT_DIR / 't27_seongnam_final.parquet'

EUNBI_DEAL_FILE = INPUT_DIR / 'eunbi_deal_clean.csv'
EUNBI_POP_FILE = INPUT_DIR / 'eunbi_population_clean.csv'
EUNBI_FIRM_FILE = INPUT_DIR / 'eunbi_firm_clean.csv'

SEONGJU_CREDIT_FILE = INPUT_DIR / 'seongju_credit_clean.csv'
SEONGJU_BUS_FILE = INPUT_DIR / 'seongju_bus_dong_clean_final.csv'
SEONGJU_SUBWAY_FILE = INPUT_DIR / 'seongju_subway_clean_final.csv'
SEONGJU_TRAFFIC_FILE = INPUT_DIR / 'seongju_traffic_accessibility_clean.csv'

KEY_CANDIDATES = ['기준년월', '행정동코드', '법정동코드', '블록코드', '행정동명']
PREFERRED_KEY_OPTIONS = [
    ['기준년월', '행정동코드'],
    ['기준년월', '블록코드'],
    ['기준년월', '법정동코드']
]

PRIMARY_DATE_COL = '기준년월'
PRIMARY_REGION_CODE_COL = '행정동코드'
PRIMARY_REGION_NAME_COL = '행정동명'
TARGET_COL = 'risk_label'              # 실제 target 컬럼이 있으면 이름을 수정하세요.
RISK_PROXY_COL = 'land_price_growth'   # 실제 위험도 proxy 컬럼이 다르면 수정하세요.

MASTER_TABLE_FILE = OUTPUT_DIR / 'master_table.csv'
STATS_RESULT_FILE = OUTPUT_DIR / 'stats_result.csv'
MODEL_COMPARE_FILE = OUTPUT_DIR / 'model_compare.csv'
DASHBOARD_EXPORT_FILE = OUTPUT_DIR / 'dashboard_export.csv'
FINAL_SUMMARY_FILE = OUTPUT_DIR / 'final_summary.md'

print('PROJECT_ROOT:', PROJECT_ROOT.resolve())
print('INPUT_DIR    :', INPUT_DIR.resolve())
print('OUTPUT_DIR   :', OUTPUT_DIR.resolve())
print('FIG_DIR      :', FIG_DIR.resolve())

## 3. 입력 파일 목록 정리

아래 표는 개인별 결과물을 최종 통합 분석에 연결하기 위한 기준표입니다. 실제 파일명이 다르면 위 환경 설정 셀에서 먼저 수정하세요.

| 담당자 | 파일명 | 설명 | 공통키 후보 | 사용 여부 |
|---|---|---|---|---|
| 근수 | geunsu_store_clean.csv | 가맹점 구조 변수 | 기준년월, 행정동코드 | O |
| 근수 | geunsu_sales_clean.csv | 매출 변수 | 기준년월, 행정동코드 | O |
| 근수 | geunsu_landprice_clean.csv | 공시지가 변수 | 기준년월 또는 연도, 행정동코드 | O |
| 지륜 | t13_seongnam_final.parquet | 이동량 구조 | 기준년월, 행정동코드 | O |
| 지륜 | t25_seongnam_final.parquet | 유입/유출 구조 | 기준년월, 행정동코드 | O |
| 지륜 | t26_seongnam_final.parquet | 체류 특성 | 기준년월, 행정동코드 | O |
| 지륜 | t27_seongnam_final.parquet | 이동수단+체류 | 기준년월, 행정동코드 | O |
| 은비 | eunbi_deal_clean.csv | 부동산 거래량 | 기준년월, 행정동코드 | O |
| 은비 | eunbi_population_clean.csv | 인구 구조 | 기준년월, 행정동코드 | O |
| 은비 | eunbi_firm_clean.csv | 기업 변수 | 기준년월, 행정동코드 | O |
| 성주 | seongju_credit_clean.csv | 신용/경제활동 | 기준년월, 행정동코드 | O |
| 성주 | seongju_bus_dong_clean_final.csv | 버스 접근성 | 기준년월, 행정동코드 | O |
| 성주 | seongju_subway_clean_final.csv | 지하철 접근성 | 기준년월, 행정동코드 | O |
| 성주 | seongju_traffic_accessibility_clean.csv | 교통 접근성 종합 | 기준년월, 행정동코드 | O |

In [ ]:
# ==================================================
# 4. 입력 파일 존재 여부 확인
# ==================================================
# 초보자 안내:
# - 이 셀은 지정한 파일이 실제로 data 폴더에 있는지 먼저 점검합니다.
# - 파일이 없으면 아래 단계에서 읽기 오류가 나므로, 가장 먼저 확인하는 것이 좋습니다.
# - OK가 출력되면 경로가 맞는 것입니다.

file_registry = {
    'geunsu_store': GEUNSU_STORE_FILE,
    'geunsu_sales': GEUNSU_SALES_FILE,
    'geunsu_landprice': GEUNSU_LANDPRICE_FILE,
    'jiryun_t13': JIRYUN_T13_FILE,
    'jiryun_t25': JIRYUN_T25_FILE,
    'jiryun_t26': JIRYUN_T26_FILE,
    'jiryun_t27': JIRYUN_T27_FILE,
    'eunbi_deal': EUNBI_DEAL_FILE,
    'eunbi_pop': EUNBI_POP_FILE,
    'eunbi_firm': EUNBI_FIRM_FILE,
    'seongju_credit': SEONGJU_CREDIT_FILE,
    'seongju_bus': SEONGJU_BUS_FILE,
    'seongju_subway': SEONGJU_SUBWAY_FILE,
    'seongju_traffic': SEONGJU_TRAFFIC_FILE,
}

missing_files = []
for name, path in file_registry.items():
    if path.exists():
        print(f'[OK] {name:<18} -> {path}')
    else:
        print(f'[MISSING] {name:<18} -> {path}')
        missing_files.append((name, str(path)))

if missing_files:
    print('\n[확인 필요] 아래 파일이 없습니다.')
    for name, path in missing_files:
        print(f'- {name}: {path}')
else:
    print('\n모든 입력 파일이 확인되었습니다.')

## 5. 개인별 파일 불러오기

이 단계에서는 개인별 clean 파일을 DataFrame으로 읽어옵니다. csv는 `pd.read_csv`, parquet는 `pd.read_parquet`를 사용합니다. 필요하면 xlsx도 같은 구조로 추가할 수 있습니다.

### 초보자 안내
- 파일이 하나라도 깨져 있거나 경로가 다르면 try/except가 오류 위치를 알려줍니다.
- 컬럼명 양쪽 공백은 병합 오류를 자주 만들기 때문에 로드 직후 바로 제거합니다.
- 아래 DataFrame 이름은 이후 셀에서 그대로 사용하므로 바꾸지 않는 것이 좋습니다.

In [ ]:
def load_any_table(path: Path, **kwargs):
    """파일 확장자에 따라 csv / parquet / xlsx를 읽는 공통 함수"""
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path, **kwargs)
    if suffix == '.parquet':
        return pd.read_parquet(path, **kwargs)
    if suffix in {'.xlsx', '.xls'}:  # xlsx 예시 포함
        return pd.read_excel(path, **kwargs)
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')

def clean_colnames(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [str(col).strip() for col in result.columns]
    return result

def safe_load_dataframe(path: Path, df_name: str) -> pd.DataFrame:
    try:
        if not path.exists():
            print(f'[SKIP] {df_name}: 파일이 없어 빈 DataFrame으로 생성합니다 -> {path}')
            return pd.DataFrame()

        temp = load_any_table(path)
        temp = clean_colnames(temp)
        print(f'[LOAD] {df_name}: shape={temp.shape}')
        display(temp.head(3))
        return temp
    except Exception as exc:
        print(f'[ERROR] {df_name}: {exc}')
        return pd.DataFrame()

df_geunsu_store = safe_load_dataframe(GEUNSU_STORE_FILE, 'df_geunsu_store')
df_geunsu_sales = safe_load_dataframe(GEUNSU_SALES_FILE, 'df_geunsu_sales')
df_geunsu_landprice = safe_load_dataframe(GEUNSU_LANDPRICE_FILE, 'df_geunsu_landprice')

df_jiryun_t13 = safe_load_dataframe(JIRYUN_T13_FILE, 'df_jiryun_t13')
df_jiryun_t25 = safe_load_dataframe(JIRYUN_T25_FILE, 'df_jiryun_t25')
df_jiryun_t26 = safe_load_dataframe(JIRYUN_T26_FILE, 'df_jiryun_t26')
df_jiryun_t27 = safe_load_dataframe(JIRYUN_T27_FILE, 'df_jiryun_t27')

df_eunbi_deal = safe_load_dataframe(EUNBI_DEAL_FILE, 'df_eunbi_deal')
df_eunbi_pop = safe_load_dataframe(EUNBI_POP_FILE, 'df_eunbi_pop')
df_eunbi_firm = safe_load_dataframe(EUNBI_FIRM_FILE, 'df_eunbi_firm')

df_seongju_credit = safe_load_dataframe(SEONGJU_CREDIT_FILE, 'df_seongju_credit')
df_seongju_bus = safe_load_dataframe(SEONGJU_BUS_FILE, 'df_seongju_bus')
df_seongju_subway = safe_load_dataframe(SEONGJU_SUBWAY_FILE, 'df_seongju_subway')
df_seongju_traffic = safe_load_dataframe(SEONGJU_TRAFFIC_FILE, 'df_seongju_traffic')

## 3-1. 개인별 통합 정리 노트북 취합

아래 섹션은 네 명의 개인 통합 정리 노트북에서 마크다운 설명, 처리 기준, 결과 요약, 변수 연결표를 원본 의미가 바뀌지 않도록 옮긴 부분이다. 원본 노트북은 수정하지 않았다.

| 담당자 | 원본 노트북 | 취합 방식 |
|---|---|---|
| eunbi | `eunbi/은비_통합정리_완성본.ipynb` | 마크다운 정리 내용 취합 |
| Geunsu | `Geunsu/개인_통합정리_템플릿_근수.ipynb` | 마크다운 정리 내용 취합 |
| Jiryun | `Jiryun/개인_통합정리_지륜템플릿_작성본.ipynb` | 마크다운 정리 내용 취합 |
| sungju | `sungju/개인_통합정리_성주.ipynb` | 마크다운 정리 내용 취합 |


## 개인별 정리: eunbi / 은비_통합정리_완성본.ipynb



## 은비 개인 통합 정리 노트북 (완성본)

- **프로젝트명**: 성남시 젠트리피케이션 분석
- **담당자명**: 은비
- **담당 파트**: 상업용 부동산 거래량 / 인구 / 신규 기업 데이터 전처리 및 상업용 부동산 거래량 / 신규 기업 / 카드(매출금) 데이터 EDA
- **작성일**: 2026-04-29
- **본 노트북 목적**: 전처리 및 EDA 내용을 팀원에게 공유하기 위해 기록을 남긴다.
- **최종 산출물 한 줄 요약**: 거래량 데이터를 법정동·월 단위 clean 파일로, 인구 / 기업 / 매출 데이터를 행정동·월 단위 clean 파일로 만들고 거래량 데이터에 대한 법정동·분기 단위 EDA 결과와 기업 / 매출 데이터에 대한 행정동·분기 단위 EDA 결과를 정리한다.


### 1. 작업 개요

- **내가 맡은 데이터/업무**
  - (1) 국토교통부 실거래가공개시스템 상업/업무용 부동산 매매거래 데이터 → 거래량 변수 추출
  - (2) 행정동 단위 인구/세대수 데이터 → 배후 수요 변수 추출
  - (3) 행정동 단위 신규 법인(기업) 데이터 → 상권 변화 신호 변수 추출
  - (4) 카드 매출 데이터 → 행정동별 매출 흐름 분석
- **왜 필요한가**: 성남시(분당구·수정구·중원구) 내 젠트리피케이션 위험 지역을 식별하기 위해, 부동산 가치 변화 / 배후 수요 / 상권 활동을 동일 단위에서 비교해야 한다.
- **최종적으로 남길 결과물**
  - clean 파일: `실거래가_2차_전처리.csv`, `population_total.csv` 정리본, `new_corp_total.csv` 정리본
  - 수준 변수 / 변화율 변수 구분표
  - 최종 변수 연결표
  - EDA 핵심 결과 요약(분기별 거래량 상위 동, 신규기업 증가율, 행정동별 매출 top, 도메인 조사 위험 수준 표)


### 2. 파일 분류표

| 파일명 | 구분 | 현재 역할 | 조치 계획 | 비고 |
|---|---|---|---|---|
| new_거래량_2차 전처리.ipynb | **최종본** | 거래량 데이터 결측/이상치 점검 | 본 노트북에 핵심 내용 통합 완료 | 거래량 파트 원본 |
| 인구_기업_전처리.ipynb | **최종본** | 인구/기업 데이터 점검 | 본 노트북에 핵심 내용 통합 완료 | 인구·기업 파트 원본 |
| EDA.ipynb | **최종본** | 거래량/신규기업/매출 EDA | 본 노트북 8장으로 결과 요약 | EDA 원본 |


### 3. 입력 파일 정리 + 단위 확정 칸

| 파일명 | 데이터 종류 | 시간 단위 | 공간 단위 | 사용 여부 | 비고 |
|---|---|---|---|---|---|
| transactions_total.csv | 부동산 매매 거래 | 월 (계약연월) | 시군구 + 법정동 | 사용 | SQL 1차 전처리 완료 (3,060행 × 9컬럼) |
| population_total.csv | 인구 / 세대수 | 월 (날짜) | 행정동 + 행정동 코드 | 사용 | SQL 1차 전처리 완료 (1,800행 × 6컬럼) |
| new_corp_total.csv | 신규 법인 수 (업종별) | 월 (date) | 행정동 + dong_code | 사용 | SQL 1차 전처리 완료(3,367행 × 10컬럼) |
| 전처리완료_카드(매출).csv | 카드 매출/거래 건수 | 일 (ta_ymd) | 행정동(admi_cty_no) | 사용 | 근수님 제공(1·2차 전처리 담당)|
| seongnam_dong_master.csv | 행정동 코드-이름 매핑 | - | 행정동 | 사용 | 매출 → 동명 매핑용, 길래 튜터님 제공 | 

#### 시간 단위 확정
- **최종 시간 단위: 분기별** 

#### 공간 단위 확정
- **최종 공간 단위: 행정동** — 거래량 데이터는 법정동 기준이므로, 분석 단계에서 법정동↔행정동 매핑을 별도로 확인해야 한다.
- 분석 대상 구: 성남시 **분당구 / 수정구 / 중원구** 3개 구.


### 4. 환경 설정 및 데이터 로드


### 5. 데이터 기본 점검 결과 요약

각 데이터셋에 대해 `info()`, `describe()`, 결측치, 중복행을 점검한 결과를 표로 정리한다.

#### 5-1. 거래량 (`transactions_total.csv`)
- **shape**: 3,060행 × 9컬럼
- **컬럼**: 시군구, 법정동, 유형, 용도지역, 전용/연면적(㎡), 건축물주용도, 거래금액(만원), 층, 계약연월
- **중복행**: 0건 (MySQL 1차 전처리에서 DISTINCT 처리)
- **결측치**
  - 시군구, 법정동, 유형, 용도지역, 전용/연면적, 건축물주용도, 거래금액, 계약연월: 0개 (0.0%)
  - 층: **1,408개 (46.01%)**
- **유형 분포**: 집합 2,871건 / 일반 189건
- **용도지역 분포 (상위)**: 중심상업 679 / 준주거 587 / 일반상업 584 / 근린상업 484 / 제3종일반주거 332 …
- **건축물주용도 분포**: 제2종근린생활 1,075 / 제1종근린생활 883 / 업무 456 / 판매 305 / 기타 204 / 교육연구 126 / 숙박 11

#### 5-2. 인구 (`population_total.csv`)
- **shape**: 1,800행 × 6컬럼
- **컬럼**: 행정구역, 행정동 코드, 날짜, 총인구수, 세대수, 세대당_인구
- **중복행**: 0건
- **결측치**: 모든 컬럼 0개
- **타입 이슈**: `총인구수`, `세대수`가 콤마 포함 문자열로 저장됨 → 정수 변환 필요
- **describe (변환 후)**
  - 총인구수: min 2,911 / 평균 18,310 / max 46,396
  - 세대수: min 1,523 / 평균 8,196 / max 18,849
  - 세대당_인구: min 1.54 / 평균 2.23 / max 3.28

#### 5-3. 신규 기업 (`new_corp_total.csv`)
- **shape**: 3,367행 × 10컬럼
- **컬럼**: date, sido_nm, sigun_nm, admi_nm, dong_code, induty_pri_cd, induty_pri_nm, induty_med_cd, induty_med_nm, ncr_crp_comp_cn
- **중복행**: 0건 (`date + dong_code + induty_pri_nm + induty_med_nm` 기준)
- **결측치**: 모든 컬럼 0개
- **describe**: ncr_crp_comp_cn min 1 / 평균 1.28 / max 12 (행정동·월·업종별 신규 법인 수)


### 6. 문제 데이터 정리 표 + 처리 기준 문서화

#### 6-1. 문제 데이터 정리 표

| 데이터셋 | 컬럼명 | 문제 유형 | 처리 방식 | 이유 | 확인 상태 |
|---|---|---|---|---|---|
| 거래량 | 층 | 결측치 46.01% (1,408건) | (일반) NaN → `whole_building`, (집합) NaN → `unknown` | 일반 유형은 통건물 매매로 층 구분 자체가 없음. 집합 유형 결측은 지하/구분 모호 → 거래금액 이상치 판단용 보조지표 수준이라 라벨로 대체. 튜터 답변 기준. | **완료** |
| 거래량 | 거래금액(만원) | 최솟값 200만원(2백만원) ~ 최댓값 198,204,140만원(약 1.98조원). 상업용 200만원은 비정상적으로 작음 | 상·하위 10% 중에서도 IQR 기반 극단값 추출(상위: Q3+1.5·IQR, 하위: Q1−0.5·IQR) | 보수적으로 봐도 1,000~1,500만원이 합리적 최소선. 단순 절대값 컷보다 분포 기반 검토. | **완료**(추출), 제거 여부 추가 검토 |
| 거래량 | 전용/연면적(㎡) | 최솟값 4㎡(약 1.21평) ~ 최댓값 197,237㎡(약 5.97만평). 면적이 작을수록 금액도 비정상적으로 낮음 | 거래금액-면적 상관관계 + 분포 보면서 이상치 컷 기준 결정 예정 | 단일 임계값으로 자르기 어려움. 두 변수 함께 봐야 함. | **검토 중** |
| 인구 | 총인구수, 세대수 | 콤마 포함 문자열 → 숫자 연산 불가 | `str.replace(',', '').astype(int)` | 집계/시각화 위해 정수 타입 필요. | **완료** |
| 매출 | ta_ymd | 일 단위 데이터 | 분기 단위로 집계(`STRFTIME('%Y-%m')`) | 다른 데이터셋과 단위 통일. | **완료**(EDA 단계) |
| 매출 | admi_cty_no | 8자리 행정동 코드 | `seongnam_dong_master.csv`의 `dong_code_8`과 LEFT JOIN | 동명 매핑 후 해석 가능. | **완료** |

#### 6-2. 처리 기준 문서화
- **원본 유지 여부**: 원본 csv는 수정하지 않고, 분석용 clean 파일을 별도로 저장한다.
- **집계 기준**: 최종 분석 단위는 **행정동 × 분기**로 통일한다. 거래량은 법정동 → 행정동 매핑 후 집계.
- **변화율 계산 기준**: 동일 공간 단위 내 전기(전월) 대비 변화율을 우선 사용한다.
- **이상치 처리 기준**: 단순 절대값 컷이 아니라 분포(IQR) 및 보조 변수(면적-금액)와의 관계를 함께 보고 판단한다.
- **결측치 처리 기준**: 의미가 명확한 결측은 의미 있는 라벨(`whole_building`, `unknown`)로 대체. 의미 불명 결측은 분석에서 제외 또는 별도 표기.


### 7. 전처리 실행 코드 (요약)

원본 노트북(`new_거래량_2차 전처리.ipynb`, `인구_기업_전처리.ipynb`)의 핵심 처리 로직을 한 곳에 모은다.


### 8. EDA 주요 결과 요약

원본 `EDA.ipynb`의 결과를 카테고리별로 정리.

#### 8-1. 거래량 변화 (분기별 법정동 거래율)
- 2023Q1 ~ 2025Q4 총 12분기 동안 분기별 거래량 비율 상위 12개 법정동을 추출.
- **전체 12분기 모두 상위에 든 법정동 (8개)**: (분당구) 서현동, 구미동, 정자동, 야탑동, 삼평동, 수내동, 금곡동 / (수정구) 창곡동
- 그 외: 성남동(중원구) 10회, 상대원동(중원구) 8회, 이매동·운중동·신흥동·백현동·대장동 등.
- 시각화: 8개 법정동의 분기별 거래 비율 변화 라인 차트.

#### 8-2. 거래량 도메인 조사 (위험 수준)

| 지역명 | 위험 수준 | 위험 유형 | 핵심 |
|---|---|---|---|
| 삼평동 | 매우 높음 | 업무/상업 복합 | 판교테크노밸리 중심, 거대 자본 위주 상권 재편 |
| 정자동 | 높음 | 상업 젠트리피케이션 | 카페거리 고급화, 소규모 진입 장벽 ↑ |
| 백현동 | 높음 | 상업 젠트리피케이션 | 판교역 인근 카페거리 브랜드화 |
| 서현동 | 보통/주의 | 상업 고착화 | 분당 최대 노후 상권, 업종 교체 주기 가팔라짐 |
| 수내동 | 보통 | 주거/상업 안정 | 학원가 연계 업종 임대료 경쟁 |
| 야탑동 | 보통 | 교통 요충지 | 터미널·병원 유동인구로 임대료 견고 |
| 창곡동 | 안정/유지 | 위례신도시 | 초기부터 높은 임대료, 조정기 |
| 금곡동 | 낮음/안정 | 주거 중심 | 미금역 인근 외 안정적 |
| 구미동 | 낮음/안정 | 주거 중심 | 분당 남단 성숙 주거지 |

→ 거래량과 연관성이 예상되는 변수: **공시지가 상승률(임대료), 프랜차이즈 침투율, 업종 교체 주기, 유동인구, 공실률**

#### 8-3. 구별 용도지역 / 건축물주용도 분포
- 분당구·수정구·중원구 3개 구의 용도지역(중심상업/준주거/일반상업/근린상업/주거지역 등) 분포를 막대그래프로 비교.
- 분당구·수정구·중원구의 건축물주용도(근린생활/업무/판매/교육연구/숙박/기타) 분포 비교.

#### 8-4. 신규 기업 증가율
- `new_corp_total.csv`를 `date × admi_nm` 피벗 후 `pct_change()`로 행정동별 신규 법인 수 월별 증가율 산출.
- 누적 신규 법인 수 상위 5개 행정동의 증가율 추이 라인 차트.

#### 8-5. 매출 변화율 (카드 매출)
- **지역별(구) 총매출액 / 총거래건수**: 분당구(신도심) 41135, 수정구(원도심) 41133, 중원구(원도심) 41131 비교.
- **월별 매출액 / 거래건수 추이**: 3개 구 라인 차트, 매출액(십억원), 거래건수(만건) 단위.
- **2025년 구별 행정동 매출 Top 5**
  - 분당구: 정자1동 65.1조원 / 백현동 17.0조원 / 정자3동 7.7조원 / 서현1동 6.1조원 / 수내1동 6.0조원
  - 중원구: 상대원1동 3.0조원 / 도촌동 2.0조원 / 성남동 0.40조원 / 하대원동 0.16조원 / 금광2동 0.11조원
  - 수정구: 위례동 0.49조원 / 태평4동 0.22조원 / 수진2동 0.17조원 / 신흥3동 0.15조원 / 시흥동 0.14조원
- **원도심(중원·수정) 행정동 Top 10**: 상대원1동, 도촌동, 위례동, 성남동, 태평4동, 수진2동, 하대원동, 신흥3동, 시흥동, 신흥2동.

#### 8-6. 매출 도메인 조사 (위험 수준 종합)

| 구 | 매우 높음 | 높음 | 보통/주의 | 낮음/안정 |
|---|---|---|---|---|
| 분당구 | 백현동 | 정자1동 | 서현1동 | 수내1동, 정자3동 |
| 중원구 | 성남동 | 금광2동 | 상대원1동 | 하대원동, 도촌동 |
| 수정구 | 신흥3동 | 수진2동, 태평4동 | 성남동(수정구) | 위례동(안정), 시흥동(특수: 토지 가격 급등) |

**핵심 인사이트**: 분당 지역은 매출 절대 규모가 매우 크므로, **객단가(amt/cnt)** 변화율로 보면 젠트리피케이션 속도를 더 명확히 볼 수 있음.


### 9. 전처리 결과 요약표 (데이터셋별)

#### 거래량
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| transactions_total.csv | 실거래가_2차_전처리.csv | 2023-01 ~ 2025-12 | 3,060 | 시군구, 법정동, 계약연월, 거래금액(만원), 전용/연면적(㎡), 용도지역, 건축물주용도, 층 | 투자/거래 활동 파악 |

#### 인구
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| population_total.csv | (변환 후 동일) | 2023-01 ~ 2025-12 | 1,800 | 행정구역, 행정동 코드, 날짜, 총인구수, 세대수, 세대당_인구 | 배후 수요 파악 |

#### 기업
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| new_corp_total.csv | (그대로 사용) | 2023-01 ~ 2025-12 | 3,367 | date, admi_nm, dong_code, induty_pri_nm, induty_med_nm, ncr_crp_comp_cn | 상권 변화 신호 파악 |

#### 매출
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| 전처리완료_카드(매출).csv | (그대로 사용, EDA 단계 집계) | 2023-01 ~ 2025-12 | (대용량) | ta_ymd, cty_rgn_no, admi_cty_no, card_tpbuz_*, amt, cnt | 소비 활동 / 객단가 변화 |


### 10. 수준 변수 / 변화율 변수 구분표 + 최종 변수 연결표

#### 수준 변수 / 변화율 변수
| 구분 | 변수명 | 의미 | 해석 |
|---|---|---|---|
| 수준 | transaction_cnt | 행정동·월 거래 건수 | 투자/거래 활동 수준 |
| 수준 | transaction_amt | 행정동·월 평균/합계 거래금액 | 부동산 가치 수준 |
| 수준 | population_total | 총인구 | 배후 수요 규모 |
| 수준 | household_cnt | 세대수 | 배후 수요 단위 수 |
| 수준 | new_corp_cnt | 신규 법인 수 | 상권 신규 진입 활동 |
| 수준 | sales_amt | 카드 매출액 | 소비 활동 수준 |
| 수준 | sales_per_cnt (객단가) | amt / cnt | 단가 수준 (젠트리피케이션 속도 핵심) |
| 변화율 | transaction_cnt_growth | 거래 건수 전월 대비 변화율 | 거래 활성화/위축 신호 |
| 변화율 | population_total_growth | 인구 변화율 | 인구 유출입 |
| 변화율 | new_corp_cnt_growth | 신규 법인 변화율 | 상권 변화 속도 |
| 변화율 | sales_per_cnt_growth | 객단가 변화율 | 가격 상승 압력 |

#### 최종 변수 연결표
| 데이터셋 | 원본 컬럼 | 최종 변수명 | 의미 | 사용 방향 |
|---|---|---|---|---|
| 거래량 | (집계) 행 수 | transaction_cnt | 거래 활동 수준 | 투자 유입 압력 |
| 거래량 | 거래금액(만원) | transaction_amt | 거래 금액 수준 | 부동산 가치 수준 |
| 거래량 | 용도지역 | zone_type | 용도지역 코드 | 상업화 단계 해석 |
| 거래량 | 건축물주용도 | bldg_use | 건축물 용도 | 업종 변화 해석 |
| 인구 | 총인구수 | population_total | 인구 | 배후 수요 |
| 인구 | 세대수 | household_cnt | 세대수 | 가구 단위 수요 |
| 인구 | 세대당_인구 | persons_per_household | 가구당 인구 | 가구 구성 변화 |
| 기업 | ncr_crp_comp_cn | new_corp_cnt | 신규 법인 수 | 상권 신규 진입 |
| 기업 | induty_pri_nm | industry_major | 산업 대분류 | 업종 변화 |
| 매출 | amt | sales_amt | 매출액 | 소비 활동 |
| 매출 | cnt | sales_cnt | 거래 건수 | 소비 빈도 |
| 매출 | amt/cnt | sales_per_cnt | 객단가 | 가격 상승 압력 |

#### 젠트리피케이션 해석 칸
- **투자 유입 압력**: transaction_cnt, transaction_cnt_growth (특히 분당 8개 동·창곡동·성남동 모니터링)
- **부동산 가치 수준**: transaction_amt (용도지역·건축물주용도 통제 후 비교)
- **가격 상승 압력**: sales_per_cnt_growth (분당 백현동·정자1동 우선)
- **배후 수요**: population_total, household_cnt (재개발 진행 중인 신흥3동·수진2동·태평4동 변화 주목)
- **상권 변화 신호**: new_corp_cnt_growth, 산업 대분류별 신규 법인 비중 변화


### 11. 최종 체크리스트 + 마지막 요약

#### 체크리스트
- [x] 최종본 / 백업 / 삭제예정 파일 구분 완료
- [x] 거래량 전처리 결과 정리 완료 (층 결측치, 이상치 추출)
- [x] 인구 전처리 결과 정리 완료 (타입 변환)
- [x] 기업 데이터 점검 완료 (결측·중복 0)
- [x] 매출 EDA 결과 정리 완료 (지역/월/행정동/원도심 Top10)
- [x] 시간 단위(월) / 공간 단위(행정동) 확정 완료
- [x] 수준 변수 / 변화율 변수 구분표 작성 완료
- [x] 최종 변수 연결표 작성 완료
- [x] 젠트리피케이션 해석 칸 작성 완료
- [ ] 거래량 면적 이상치 컷 기준 확정 (검토 중)
- [ ] 거래량 법정동 ↔ 행정동 매핑 확정
- [ ] 거래량·인구·기업·매출 행정동·월 단위 통합 marts 파일 저장

#### 팀 공유용 5줄 요약
1. 성남시 분당·수정·중원 3개 구의 부동산 거래량(3,060건) / 인구(1,800행) / 신규 기업(3,367행) / 카드 매출 데이터를 행정동·월 기준으로 정리했습니다.
2. 거래량의 핵심 결측은 `층`(46%)이며 유형(일반/집합)에 따라 `whole_building` / `unknown`으로 라벨링했고, 거래금액·면적 이상치는 IQR 기반으로 추출해 별도 검토 중입니다.
3. 분기별 거래량 상위 동(분당구 7개 + 수정구 창곡동) 8곳을 식별했고, 도메인 조사 결과 삼평·정자·백현이 매우 높음/높음 위험으로 분류됩니다.
4. 매출 EDA에서는 분당 백현동·정자1동, 수정구 신흥3동·수진2동·태평4동, 중원구 성남동·금광2동이 젠트리피케이션 위험 핵심지로 확인됐습니다.
5. 다음 단계는 거래량 면적 이상치 컷 확정, 법정동↔행정동 매핑, 그리고 4개 데이터셋의 행정동·월 단위 통합 marts 파일 산출입니다.


**취합 메모**: `eunbi/은비_통합정리_완성본.ipynb`에서 마크다운 셀 12개를 취합했다. 실행 코드와 산출물 생성 로직은 원본 노트북 및 아래 최종 분석 템플릿 코드 셀을 함께 참조한다.


## 개인별 정리: Geunsu / 개인_통합정리_템플릿_근수.ipynb



## 개인(근수) 통합 정리용 Notebook

- 프로젝트명: [금융] 젠트리피케이션 위험도 분석 시스템 구축 및 상생 체계 제안
- 분석 대상: 성남시 원도심 (수정구 41133 + 중원구 41131), 분당구 41135는 비교 대조군
- 담당자명: 근수
- 담당 파트: 가맹점 / 매출 / 공시지가
- 작성일: 2026-04-29
- 이 노트북의 목적: 근수 파트의 1차 전처리 결과, 실제 산출물, EDA 완료/미완료 상태를 한곳에 정리하고 최종 분석에 투입할 변수 세트를 확정
- 최종 산출물 한 줄 요약: 행정동(admi_cty_no) × 분기 단위 통합 변수표를 만들되, 현재 확정 산출물은 가맹점 EDA 지표(`open_rate`, `close_rate`, `franchise_ratio`, `industry_diversity_H`, `survival_rate_1y`)이며 `sales_amt`, `avg_land_price`, `land_price_growth`는 후속 EDA/집계 후 결합


### 1. 작업 개요

- 담당 데이터/업무: 가맹점(카드사 가맹점 통합), 매출, 공시지가 원본 -> 행정동×분기 단위 위험도 지표화
- 왜 필요한지: 본 프로젝트의 4대 핵심 지표(개·폐업률 / 프랜차이즈 침투율 / 업종 다양성 / 신생 점포 잔존율)와 부동산 압력(공시지가), 매출 활력 축이 최종 위험도 변수에 포함되기 때문
- 분석 범위: 2023-01 ~ 2025-12 (36개월), 성남시 50개 행정동, 시군구 3개(41131 중원·41133 수정·41135 분당), 블록코드 5,442개
- 분석 단위: 원도심(41131+41133) vs 비교 대조군(41135 분당구) 양극화 분석 구도
- 최종적으로 남길 파일/변수/표:
  - 전처리 완료 데이터: 전처리완료_카드(가맹점).csv, 전처리완료_부동산(공시지가).csv, 전처리완료_카드(매출).csv
  - EDA 산출 완료: 가맹점 4개 지표 + 종합 단계(탐색용 5단계)
  - 후속 산출 필요: 매출 EDA, 공시지가 EDA, 행정동×분기 통합 변수표
  - 최종 변수 연결표 1개


### 2. 삭제/백업/최종본 노트북 정리

현재 `Geunsu` 폴더 기준으로 실제 존재하는 노트북과 작성 필요 항목을 구분합니다.

| 파일명 | 현재 역할 | 유지 여부 | 비고 |
|---|---|---|---|
| 1차_전처리_가맹점.ipynb | 가맹점 1차 전처리 | 유지 | 완료, `전처리완료_카드(가맹점).csv` 산출 |
| 1차_전처리_매출.ipynb | 매출 1차 전처리 | 유지 | 완료, 최종 87,263,128행. 저장 경로는 `E:\전처리완료_카드(매출).csv` |
| 1차_전처리_공시지가.ipynb | 공시지가 1차 전처리 | 유지 | 완료, `전처리완료_부동산(공시지가).csv` 산출 |
| EDA_가맹점.ipynb | 전처리된 가맹점 데이터 탐색 | 유지 | 완료, 4대 핵심 지표 + 탐색용 5단계 분류 산출 |
| EDA_매출.ipynb | 매출 EDA | 작성 필요 | 은비님 담당 |
| EDA_공시지가.ipynb | 공시지가 EDA | 작성 필요 | 지륜님 담당 |
| 개인_통합정리_템플릿_근수.ipynb | 근수 파트 통합 정리 | 유지 | 실제 파일/산출물 상태 반영용 |


### 3. 문제 데이터 정리 섹션

1차 전처리 과정에서 실제로 발견·처리했거나 후속 처리가 필요한 이슈 기록입니다.

| 데이터셋 | 컬럼명 | 문제 유형 | 처리 방식 | 이유 | 확인 상태 |
|---|---|---|---|---|---|
| 가맹점 | ta_ym | int64 형식 (YYYYMM) | `pd.to_datetime(format='%Y%m')` 변환 | 분기 집계·시계열 분석을 위한 datetime 통일 | 완료 (range 2023-01 ~ 2025-12) |
| 가맹점 | cty_rgn_no / admi_cty_no / blk_cd | int64 -> 코드성 데이터 | `astype(str)` 변환 | 앞자리 0 보존, 조인 키 일관성 | 완료 |
| 가맹점 | sale | NULL 387,860건 (30억 초과 또는 국세청 미등록) | `'E'` 구간으로 채움 | 데이터 정의서상 의미 있는 결측 -> 신규 등급으로 분리 보존 | 완료 |
| 가맹점 | mm_cnt | 최댓값 134,368개월(약 1만년) — 비현실적 | 제거하지 않고 보존 (블록 단위 누적값으로 추정) | 동일 블록·업종 시계열에서 일관 추세 확인됨 | 검토 완료, 분석 시 행정동 단위 집계로 희석 |
| 가맹점 | open_cnt > mer_cnt | 11건 (mer_cnt=0인데 open_cnt>=1) | 보존 | 명세서: 신규 등록은 가맹점주 변경 재신청 포함 -> 정의상 가능 | 완료 |
| 가맹점 | stop_cnt > mer_cnt | 17건 | 보존 | 휴업 시점과 가맹점 수 집계 시점 차이 가능 | 완료 |
| 가맹점 | close_cnt > mer_cnt | 6,203건 | 분기 합산 후 비율 산출, 분모 안정성 필터 적용 | 월별 블록·업종 단위에서는 폐업 수가 현재 가맹점 수를 넘을 수 있음 | EDA 반영, 추가 검토 가능 |
| 가맹점 | 키컬럼(8개) 중복 | 0건 | — | 키 무결성 확인 완료 | 완료 |


### 4. 처리 기준 문서화 + 근수 전용 정리 포인트

#### 처리 원칙
- 원본 유지: 원본 파일은 수정하지 않고 `전처리완료_*.csv` 형태로 별도 생성
- 결측 처리: 원본의 의미 있는 결측(예: `sale=NULL` -> 30억 초과)은 별도 카테고리로 보존(`'E'`)
- 비율 계산: 비율의 평균이 아닌 **합산 후 비율** 방식 채택 (Σopen_cnt / Σmer_cnt × 100)
- 분모 안정성 필터: 행정동×분기 mer_cnt 합 < 50 또는 open_11_cnt(t-12) < 5 인 조합은 비율 지표에서 제외
- 분기 단위: `ta_ym.dt.to_period("Q")` 변환 후 집계

#### 공시지가 산/공원/비주거성 필지 처리 기준
- 현재 clean 파일 확인 컬럼: `법정동코드`, `법정동명`, `기준연도`, `기준월`, `기준년월`, `건물용도분류명`, `공시지가`
- 현재 clean 파일에는 `지목명` / `용도지역` 컬럼이 없어 산·공원·도로·하천 등 비상업 필지 제외 기준을 확정하기 어렵다.
- 우선 처리안: `건물용도분류명`으로 상업/비상업 후보를 분류하고, 판단이 불가능한 케이스는 원본 또는 sungju 공시지가 파트에서 지목·용도지역 정보를 보강한다.
- 판단이 어려운 케이스 기록 위치: `Geunsu/reference/공시지가_예외검토.csv` (현재 미작성)

#### 최종 분석 변수 — 가맹점 핵심 지표 + 매출/부동산 압력 축

| 분석 축 | 대표 변수명 | 산식 | 단위 | 현재 상태 | 해석 방향 |
|---|---|---|---|---|---|
| 개·폐업 (활력) | `open_rate` | Σ open_cnt / Σ mer_cnt × 100 | 행정동 × 분기 | 가맹점 EDA 산출 | 높음 = 신규 진입 활발 |
| 개·폐업 (활력) | `close_rate` | Σ close_cnt / Σ mer_cnt × 100 | 행정동 × 분기 | 가맹점 EDA 산출 | 높음 = 퇴출 압력 강함 |
| 개·폐업 (활력) | `net_change` | open_rate - close_rate | 행정동 × 분기 | 가맹점 EDA 산출 | 음수 = 순감소(쇠퇴 신호) |
| 프랜차이즈 (체인화) | `franchise_ratio` | Σ fran_cnt / Σ mer_cnt × 100 | 행정동 × 분기 | 가맹점 EDA 산출 | 높음 = 프랜차이즈 침투 |
| 업종 다양성 | `industry_diversity_H` | H = -Σ(p_i × ln p_i), p_i = `card_tpbuz_nm_2`별 비율 | 행정동 × 분기 | 가맹점 EDA 산출 | 낮음 = 업종 획일화 |
| 신생 점포 잔존 | `survival_rate_1y` | open_12_23_cnt(t) / open_11_cnt(t-12) × 100 | 행정동 × 분기 (2024 Q1~) | 가맹점 EDA 산출 | 낮음 = 진입비용 충격 |

#### 단계 분류 체계 정리
- 현재 `EDA_가맹점.ipynb` 산출물은 탐색용으로 **초기 / 주의 / 경계 / 위험 / 쇠퇴** 5단계를 사용한다.


### 5. 결과 요약표와 최종 변수 연결표

#### 진행 상태 요약 (2026-04-29 기준)
| 데이터셋 | 1차 전처리 | EDA | 비고 |
|---|---|---|---|
| 가맹점 | 완료 | 완료 | `Geunsu/data/전처리완료_카드(가맹점).csv` 확인, 4대 핵심 지표 + 탐색용 5단계 분류 산출 |
| 매출 | 완료 | 필요 | 최종 87,263,128행. clean 파일은 `E:\전처리완료_카드(매출).csv` 저장으로 노트북 기록, 저장소 내 파일 미확인 |
| 공시지가 | 완료 | 필요 | `Geunsu/data/전처리완료_부동산(공시지가).csv` 확인, 행정동 매핑 및 `avg_land_price`/`land_price_growth` 산출 필요 |

#### 가맹점 전처리 결과 요약표
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| 가맹점_정보(통합).csv | 전처리완료_카드(가맹점).csv | 2023-01 ~ 2025-12 | 2,130,071 | ta_ym, cty_rgn_no, admi_cty_no, blk_cd, card_tpbuz_cd, card_tpbuz_nm_1/2, sale, mm_cnt, mer_cnt, fran_cnt, open_cnt, stop_cnt, close_cnt, open_11~60_cnt | 4대 핵심 지표(개·폐업률 / 프랜차이즈 침투율 / 업종 다양성 / 신생 점포 잔존율) 산출 |

#### 매출 전처리 결과 요약표
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| 매출_정보(통합).csv | 전처리완료_카드(매출).csv (`E:\` 저장 기록, 저장소 내 미확인) | 2023-01-01 ~ 2025-12-31 | 87,263,128 | ta_ymd, cty_rgn_no, admi_cty_no, card_tpbuz_cd, card_tpbuz_nm_1/2, hour, sex, age, day, amt, cnt | 행정동×분기 `sales_amt` 및 소비 활력 변수 산출 |

#### 공시지가 전처리 결과 요약표
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| 성남시_공시지가_통합__202604241636.csv | 전처리완료_부동산(공시지가).csv | 2023 ~ 2025 추정(기준연도/기준월 기준 확인 필요) | 15,557 | 법정동코드, 법정동명, 기준연도, 기준월, 기준년월, 건물용도분류명, 공시지가 | 행정동 매핑 후 `avg_land_price`, `land_price_growth` 산출 |

#### 핵심 데이터 분포 (가맹점 기준)
| 항목 | 값 |
|---|---|
| 시군구 분포 | 41135 분당 1,064,991 / 41133 수정 534,650 / 41131 중원 530,430 |
| 고유 행정동 수 | 50개 |
| 고유 블록코드 수 | 5,442개 |
| 업종 대분류 수 | 9개 (음식 · 소매/유통 · 생활서비스 ··· 공연/전시) |
| 업종 중분류 수 | 83개 |
| sale 분포 | A 1,069,467 / B 231,895 / C 214,109 / D 159,267 / E 387,860 / 신규 67,473 |

#### 최종 변수 연결표 (원본 컬럼 -> 분석 변수)
| 데이터셋 | 원본 컬럼 | 최종 변수명 | 의미 | 사용 지표 | 산출 상태 |
|---|---|---|---|---|---|
| 가맹점 | mer_cnt | `mer_cnt` | 가맹점 수 (분모) | 모든 비율 지표 | 완료 |
| 가맹점 | open_cnt | `open_rate` (집계 후) | 신규 가맹점 비율 | 개업률 | 완료 |
| 가맹점 | close_cnt | `close_rate` (집계 후) | 폐업 가맹점 비율 | 폐업률 | 완료 |
| 가맹점 | fran_cnt | `franchise_ratio` (집계 후) | 프랜차이즈 비율 | 침투율 | 완료 |
| 가맹점 | card_tpbuz_nm_2 | `industry_diversity_H` (Shannon) | 업종 분포 엔트로피 | 업종 다양성 | 완료 |
| 가맹점 | open_11_cnt, open_12_23_cnt | `survival_rate_1y` | 1년 잔존율 (lag-12 비교) | 신생 점포 잔존 | 완료 |
| 가맹점 | sale | `sale` (E로 NULL 채움) | 매출 등급 | 대형 가맹점 비중 보조 | 완료 |

#### 단계 분류 산출표 (현재 가맹점 EDA 스키마)
| 컬럼 | 타입 | 설명 |
|---|---|---|
| admi_cty_no | str(8) | 행정동 코드(최종 통합 시 필요) |
| 행정동명 | str | 매핑 결과 |
| 구명 | str | 분당구/수정구/중원구 |
| 개폐업률_단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |
| 프랜차이즈_단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |
| 업종다양성_단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |
| 잔존율_단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |
| 종합점수 | float | 4개 지표 단계 점수 평균(초기 0 ~ 쇠퇴 4) |
| 종합단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |


### 6. 최종 체크리스트 + 마지막 요약

#### 체크리스트
- [x] 가맹점 1차 전처리 완료 (2,130,071행, 2023-01 ~ 2025-12)
- [x] 키 컬럼 8개 기준 중복 0건 확인
- [x] sale NULL 387,860건 -> 'E' 등급 보존 처리
- [x] 시군구 코드 3개(41131/41133/41135) · 행정동 50개 · 블록 5,442개 무결성 확인
- [x] 가맹점 EDA 완료 (4대 핵심 지표 + 탐색용 5단계 분류 + 종합단계)
- [x] 매출 1차 전처리 완료 (0매출/0건수 제거 후 87,263,128행)
- [ ] 매출 clean 파일 `Geunsu/data/전처리완료_카드(매출).csv` 저장 확인
- [ ] 매출 EDA 완료 (행정동×분기 `sales_amt` 추이, 업종별 매출 변화)
- [x] 공시지가 1차 전처리 완료 (`전처리완료_부동산(공시지가).csv`, 15,557행)
- [ ] 공시지가 행정동 매핑 기준 확정
- [ ] 공시지가 산/공원/비상업 필지 제외 기준 확정 및 적용
- [ ] 공시지가 EDA 완료 (`avg_land_price`, `land_price_growth` 산출)
- [ ] 행정동×분기 통합 변수표 1개 산출 (`open_rate`, `close_rate`, `franchise_ratio`, `industry_diversity_H`, `survival_rate_1y`, `avg_land_price`, `land_price_growth`, `sales_amt`)
- [ ] clean 파일 3종 `Geunsu/data/` 또는 팀 공통 경로 저장 완료
- [ ] 노트북 중복 정리(전처리 3종 + EDA 3종 + 통합정리 1종) 완료
- [ ] 팀 최종 4단계 체계에 맞춰 가맹점 EDA의 `쇠퇴` 단계 재매핑 여부 결정

#### 현재 진행 상태 (2026-04-29)
- **완료**: 가맹점 1차 전처리 + EDA, 매출 1차 전처리, 공시지가 1차 전처리
- **확인 필요**: 매출 clean CSV의 저장소 내 위치, 공시지가 행정동 매핑 기준
- **다음 작업**: 매출·공시지가 EDA 작성 후 행정동×분기 통합 변수표 산출

#### 팀 공유용 5줄 요약
1. 성남시 가맹점 데이터 213만건을 행정동(50개) × 분기 단위로 정제하여 4대 핵심 지표(개·폐업률 / 프랜차이즈 침투율 / 업종 다양성 H / 신생 점포 잔존율)를 산출했습니다.
2. `EDA_가맹점.ipynb`는 현재 탐색용 5단계(초기/주의/경계/위험/쇠퇴)를 사용하므로, 팀 최종 4단계(초기/주의/경계/위험)와 맞추려면 `쇠퇴` 단계 재매핑 기준이 필요합니다.
3. 매출 데이터는 0매출/0건수 제거 후 87,263,128행으로 1차 전처리되었고, 저장소 내 clean 파일 위치 확인 및 행정동×분기 `sales_amt` EDA가 남아 있습니다.
4. 공시지가는 15,557행 clean 파일이 확인되었지만, 행정동 매핑과 `avg_land_price`, `land_price_growth` 산출은 아직 필요합니다.
5. 최종 후속 산출물은 `전처리완료_카드(가맹점).csv` + `전처리완료_카드(매출).csv` + `전처리완료_부동산(공시지가).csv` + 행정동×분기 통합 변수표입니다.


**취합 메모**: `Geunsu/개인_통합정리_템플릿_근수.ipynb`에서 마크다운 셀 7개를 취합했다. 실행 코드와 산출물 생성 로직은 원본 노트북 및 아래 최종 분석 템플릿 코드 셀을 함께 참조한다.


## 개인별 정리: Jiryun / 개인_통합정리_지륜템플릿_작성본.ipynb



## 지륜님 개인 통합 정리용 Notebook 작성본 - T데이터 전체

- 프로젝트명: 성남시 젠트리피케이션 예측/분석
- 담당자명: 지륜
- 담당 파트: 통신 T데이터 통합/전처리/EDA, 공시지가 상승률 EDA
- 작성일: 2026-04-29
- 정리 목적: `T4~T27` 통신 데이터와 공시지가 상승률 EDA의 산출물, 전처리 기준, 검증 결과, 변수 후보를 한 노트북에서 확인할 수 있게 정리한다.
- 최종 산출물 한 줄 요약: `T4~T27` 통신 테이블과 공시지가 2023~2025 상승률 EDA를 함께 정리한다.

> 이 노트북은 새 분석을 처음부터 다시 하는 노트북이 아니라, 지금까지 만든 통합 파일과 EDA 결과를 팀 공유용으로 재정리한 최종 기록용 노트북이다.

> 이 파일은 지륜 전용 템플릿 형식에 맞춘 상세 작성본이다. 공통 양식 요약본은 `개인_통합정리_공통템플릿_작성본_지륜.ipynb`에서 확인한다.


### 1. 작업 개요

#### 정리 범위
- 포함: `T4, T5, T6, T7, T8, T9, T10, T11, T12, T13, T14, T16, T20, T21, T22, T23, T24, T25, T26, T27`
- 현재 폴더 기준 미확보: `T15, T17, T18, T19`
- 통합 기간: 2023년 1월부터 2025년 12월까지
- 핵심 흐름: 원본/중간 통합 파일 확인 -> 날짜/지역/코드/결측 점검 -> 최종 parquet/csv 산출물 확정 -> 월별/분기별 EDA -> 모델링 변수 후보 정리

#### 기존 작업 노트북 역할
| 노트북 | 역할 | 포함 내용 |
|---|---|---|
| `1차_전처리_small.ipynb` | T4~T12, T14, T16, T20~T24 기초 전처리 | 날짜 컬럼 확인, 범주형 분포, 최종 date/final 파일 저장 |
| `1차_전처리.ipynb` | T13, T25, T26, T27 상세 전처리 | O/D 지역명 결측 보정, 99 코드 유지 판단, CNT/DURATION/코드값 분포 확인 |
| `eda.ipynb` | 월별 인구/유동 지표 EDA | T13 외부유입, T24 유동인구, 경제활동 연령층 proxy, 인구 데이터 결합 |
| `my_eda_월별EDA_백업.ipynb` | 월별 행정동 단위 통합 EDA | 전체 T데이터 구조 검토, 월별 변수 해석 메모, 후보 변수 정리 |
| `my_eda.ipynb` | 분기별 EDA 최종 후보 정리 | T24 행정동 분기 패널, T13/T22/T26 보조 지표, PURPOSE/TRANS_GB 코드 정의 반영 |
| `공시지가_test.ipynb` | 별도 공시지가 EDA | 통신 T데이터 정리 범위 밖이지만 팀 최종 분석에 결합 가능 |


### 2. 최종 입력/산출 파일 정리

아래 표는 현재 `data` 폴더에서 확인한 최종 사용 후보 파일 기준이다. `final_v2`, `date_final`, `final` 파일을 우선 산출물로 보았다.

| 테이블 | 최종 파일명 | 행 수 | 기간 범위 | 단위 | 주요 의미 | 현재 판단 |
|---|---|---:|---|---|---|---|
| T4 | `t4_2023_2025_all_date_final.parquet` | 1,211,106 | 2023-01~2025-12 | 시군구 | 도착지 + 목적 + 성/연령 | 구조 파악/보조 |
| T5 | `t5_2023_2025_all_date_final.parquet` | 43,306,619 | 2023-01-01~2025-12-31 | 행정동 | 도착지 + 목적 + 성/연령 | 구조 파악/보조 |
| T6 | `t6_2023_2025_all_date_final.parquet` | 1,927,203 | 2023-01~2025-12 | 시군구 | 도착지 + 이동수단 + 성/연령 | 구조 파악/보조 |
| T7 | `t7_2023_2025_all_date_final.parquet` | 68,932,750 | 2023-01-01~2025-12-31 | 행정동 | 도착지 + 이동수단 + 성/연령 | 구조 파악/보조 |
| T8 | `t8_2023_2025_all_date_final.parquet` | 1,638,511 | 2023-01~2025-12 | 시군구 | 출발지 + 목적 + 성/연령 | 구조 파악/보조 |
| T9 | `t9_2023_2025_all_date_final.parquet` | 50,298,816 | 2023-01-01~2025-12-31 | 행정동 | 출발지 + 목적 + 성/연령 | 구조 파악/보조 |
| T10 | `t10_2023_2025_all_date_final.parquet` | 1,971,599 | 2023-01~2025-12 | 시군구 | 출발지 + 이동수단 + 성/연령 | 구조 파악/보조 |
| T11 | `t11_2023_2025_all_date_final.parquet` | 71,457,282 | 2023-01-01~2025-12-31 | 행정동 | 출발지 + 이동수단 + 성/연령 | 구조 파악/보조 |
| T12 | `t12_2023_2025_all_final_v2.parquet` | 6,743,756 | 2023-01~2025-12 | 시군구 OD | 출발지->도착지 + 목적 | 구조 파악/보조 |
| T13 | `t13_2023_2025_all_final_v2.parquet` | 279,996,851 | 2023-01-01~2025-12-31 | 행정동 OD | 출발지->도착지 + 목적 | 핵심 후보 |
| T14 | `t14_2023_2025_all_final_v2.parquet` | 8,472,627 | 2023-01~2025-12 | 시군구 OD | 출발지->도착지 + 이동수단 | 구조 파악/보조 |
| T16 | `t16_2023_2025_all_date_final.parquet` | 1,211,106 | 2023-01~2025-12 | 시군구 | 도착지 + 목적 + 체류시간 | 구조 파악/보조 |
| T20 | `t20_2023_2025_all_date_final.csv` | 6,048 | 2023-01~2025-12 | 시군구 | 출발지 + 이동수단 + 거리/탄소 | 참고 |
| T21 | `t21_2023_2025_all_date_final.csv` | 78,840 | 2023-01-01~2025-12-31 | 시군구 | 날짜 + 시간대 | 참고, CNT 없음 |
| T22 | `t22_2023_2025_all_date_final.parquet` | 2,590,644 | 2023-01-01~2025-12-31 | 행정동 | 시간대 + 성/연령 + 내외국인 | 보조 후보 |
| T23 | `t23_2023_2025_all_date_final.parquet` | 4,693,040 | 2023-01-01~2025-12-31 | 시군구 | 시간대 + 목적 유동인구 | 구조 파악/보조 |
| T24 | `t24_2023_2025_all_date_final.parquet` | 43,306,619 | 2023-01-01~2025-12-31 | 행정동 | 시간대 + 목적 유동인구 | 핵심 후보 |
| T25 | `t25_2023_2025_all_final_v2.parquet` | 261,148,533 | 2023-01-01~2025-12-31 | 시군구 OD | 출발지->도착지 + 시간대 + 목적 + 이동수단 | 보조 후보 |
| T26 | `t26_2023_2025_all_final.parquet` | 258,637,442 | 2023-01-01~2025-12-31 | 도착 행정동 | 도착지 + 시간대 + 목적 + 이동수단 + 체류시간 | 핵심 후보 |
| T27 | `t27_2023_2025_all_final.parquet` | 278,322,883 | 2023-01-01~2025-12-31 | 출발 행정동 | 출발지 + 시간대 + 목적 + 이동수단 + 체류시간 | 선택 후보 |

#### 현재 폴더 기준 누락 테이블
| 테이블 | 상태 | 메모 |
|---|---|---|
| T15 | 파일 없음 | `data` 폴더에 최종 산출물 없음 |
| T17 | 파일 없음 | `data` 폴더에 최종 산출물 없음 |
| T18 | 파일 없음 | `data` 폴더에 최종 산출물 없음 |
| T19 | 파일 없음 | `data` 폴더에 최종 산출물 없음 |

#### 공시지가 EDA 입력 파일
| 파일명 | 행 수 | 컬럼 수 | 기간 | 주요 내용 |
|---|---:|---:|---|---|
| `2차전처리_공시지가데이터.csv` | 15,542 | 13 | 2023~2025 | 필지별 공시지가, 법정동/구, 기준연도/월 |


### 2-1. 원본/중간 파일 -> 최종 산출물 매핑

템플릿의 `원본 파일 -> 최종 parquet 파일 매핑표` 항목을 전체 T데이터 기준으로 확장했다. 현재 `data` 폴더에 남아 있는 파일과 기존 전처리 노트북의 read/write 기록을 함께 기준으로 정리했다.

| 테이블 | 전처리 기록상 입력/중간 파일 | 최종 산출물 | 정리 상태 | 비고 |
|---|---|---|---|---|
| T4 | `t4_all.parquet` | `t4_2023_2025_all_date_final.parquet` | 완료 | 날짜/범주형 확인 후 저장 |
| T5 | `t5_all.parquet` | `t5_2023_2025_all_date_final.parquet` | 완료 | 행정동 도착지 목적 데이터 |
| T6 | `t6_all.parquet` | `t6_2023_2025_all_date_final.parquet` | 완료 | 도착지 이동수단 데이터 |
| T7 | `t7_all.parquet` | `t7_2023_2025_all_date_final.parquet` | 완료 | 행정동 도착지 이동수단 데이터 |
| T8 | `t8_all.parquet` | `t8_2023_2025_all_date_final.parquet` | 완료 | 출발지 목적 데이터 |
| T9 | `t9_all.parquet` | `t9_2023_2025_all_date_final.parquet` | 완료 | 행정동 출발지 목적 데이터 |
| T10 | `t10_all.parquet` | `t10_2023_2025_all_date_final.parquet` | 완료 | 출발지 이동수단 데이터 |
| T11 | `t11_all.parquet` | `t11_2023_2025_all_date_final.parquet` | 완료 | 행정동 출발지 이동수단 데이터 |
| T12 | `t12_2023_2025_all.parquet` -> `t12_2023_2025_all_final.parquet` | `t12_2023_2025_all_final_v2.parquet` | 완료 | 시군구 OD 목적 데이터 |
| T13 | `t13_2023_2025_all.parquet` -> `clean` -> `clean_fixed` -> `final` | `t13_2023_2025_all_final_v2.parquet` | 완료 | O/D 지역명 결측 보정 및 99 코드 유지 |
| T14 | `t14_2023_2025_all.parquet` -> `t14_2023_2025_all_final.parquet` | `t14_2023_2025_all_final_v2.parquet` | 완료 | 시군구 OD 이동수단 데이터 |
| T15 | 없음 | 없음 | 현재 폴더 기준 누락 | `data` 폴더와 기존 노트북 read/write 기록에서 확인 안 됨 |
| T16 | `t16_2023_2025_all.parquet` | `t16_2023_2025_all_date_final.parquet` | 완료 | 목적 + 체류시간 데이터 |
| T17 | 없음 | 없음 | 현재 폴더 기준 누락 | `data` 폴더와 기존 노트북 read/write 기록에서 확인 안 됨 |
| T18 | 없음 | 없음 | 현재 폴더 기준 누락 | `data` 폴더와 기존 노트북 read/write 기록에서 확인 안 됨 |
| T19 | 없음 | 없음 | 현재 폴더 기준 누락 | `data` 폴더와 기존 노트북 read/write 기록에서 확인 안 됨 |
| T20 | `t20_2023_2025_all.csv` | `t20_2023_2025_all_date_final.csv` | 완료 | CSV 유지, 거리/탄소 포함 참고용 |
| T21 | `t21_2023_2025_all.csv` | `t21_2023_2025_all_date_final.csv` | 완료 | CNT 없음, 참고용 |
| T22 | `t22_2023_2025_all.parquet` | `t22_2023_2025_all_date_final.parquet` | 완료 | 성/연령/내외국인 보조 후보 |
| T23 | `t23_2023_2025_all.parquet` | `t23_2023_2025_all_date_final.parquet` | 완료 | 시군구 목적 유동인구 |
| T24 | `t24_2023_2025_all.parquet` | `t24_2023_2025_all_date_final.parquet` | 완료 | 행정동 유동인구 핵심 후보 |
| T25 | `t25_2023_2025_all.parquet` -> `clean` -> `final` | `t25_2023_2025_all_final_v2.parquet` | 완료 | 시군구 OD, O/D 결측 보정 및 99 코드 유지 |
| T26 | `t26_2023_2025_all_clean.parquet` -> `fixed` | `t26_2023_2025_all_final.parquet` | 완료 | 도착 행정동 체류/목적/수단 핵심 후보 |
| T27 | `t27_2023_2025_all.parquet` -> `clean` | `t27_2023_2025_all_final.parquet` | 완료 | 출발 행정동 체류/목적/수단 선택 후보 |

현재 최종 정리본에서 실제 분석 후보로 잡은 파일은 모두 존재한다. 단, T13/T25/T26/T27의 일부 초기 입력/중간 파일은 전처리 노트북 기록에는 남아 있지만 현재 `data` 폴더에는 최종 파일 중심으로 남아 있다.


### 2-2. 누락 여부 재점검 결과

| 점검 항목 | 결과 | 조치 |
|---|---|---|
| `T4~T27` 테이블 포함 여부 | T4~T14, T16, T20~T27 포함. T15/T17/T18/T19는 파일 없음 | 누락 테이블로 명시 |
| 최종 파일 존재 여부 | 정리본의 최종 후보 파일 20개 모두 존재 | 파일 존재 검증 코드 유지 |
| 템플릿의 매핑표 항목 | 첫 정리본에서는 파일 목록과 처리 기준에 흡수되어 있었음 | `2-1. 원본/중간 파일 -> 최종 산출물 매핑` 섹션 추가 |
| 전처리 기준 | 날짜, 지역 매핑, 99 코드, 핵심 컬럼 결측, 코드값 해석 기준 포함 | 유지 |
| EDA 정리 | 월별 EDA, 분기별 EDA, 모델링 후보 변수 포함 | 유지 |
| 통신 외 작업 | 공시지가 EDA는 통신 T데이터 범위 밖 | 작업 노트북 역할표에 별도 표시 |


### 3. 파일 존재 여부 및 메타데이터 검증

아래 코드는 대용량 parquet를 전부 읽지 않고, 가능한 한 parquet 메타데이터와 row group statistics로 행 수, 컬럼 수, 기간 범위를 확인한다. CSV는 파일 크기가 작아 직접 확인한다.


### 4. 통합 및 전처리 기준 문서화

#### 공통 처리 기준
- 원본/중간 파일은 직접 수정하지 않고, 분석용 최종 산출물을 `data` 폴더에 별도 저장했다.
- parquet 저장 가능 테이블은 parquet를 최종 후보로 두고, T20/T21처럼 크기가 작거나 기존 산출물이 CSV인 경우 CSV를 유지했다.
- 날짜 컬럼은 `ETL_YM` 또는 `ETL_YMD` 기준으로 2023~2025 전체 기간을 확인했다.
- `CNT`, `DURATION`, `PURPOSE`, `TRANS_GB`, `SEX_CD`, `AGE_GRP`는 임의 삭제하지 않고 분포와 결측 여부를 먼저 확인했다.
- 지역명/O-D 매핑 결측은 코드 기준으로 보정 가능한 값만 보정했고, 데이터 공급처의 미확인 코드로 보이는 `99`는 원본 의미 보존을 위해 유지했다.
- 대용량 테이블은 전체 스캔을 반복하지 않도록 DuckDB와 parquet 메타데이터를 사용해 검증했다.

#### 상세 처리 메모
| 구분 | 처리 내용 | 최종 판단 |
|---|---|---|
| 날짜 | `ETL_YM`, `ETL_YMD` 기준 기간 범위 확인 | 2023-01~2025-12 범위 확보 |
| 지역 매핑 | O/D 지역명, 행정동명, 좌표 결측 확인 후 코드 매핑 가능한 값 보정 | 99/미확인 코드는 유지 |
| 핵심 수치 | `CNT`, `DURATION` 분포와 이상치 확인 | 큰 값 자체를 이상치로 단정하지 않고 EDA에서 해석 |
| 코드값 | `PURPOSE`, `TRANS_GB`, `SEX_CD`, `AGE_GRP` 분포 확인 | 코드 정의와 추정값 성격을 별도 기록 |
| 저장 검증 | 최종 parquet/csv 저장 후 재오픈 및 shape 확인 | 최종 후보 파일로 사용 가능 |


### 5. 문제 데이터 및 예외 케이스 정리

| 데이터셋 | 컬럼/영역 | 확인 내용 | 처리 방식 | 이유 |
|---|---|---|---|---|
| T12 | `D_CTY_NM` | 일부 지역명 결측 | 최종 파일에 기록, 사용 시 코드/좌표 우선 확인 | 핵심 CNT/코드 컬럼은 결측 없음 |
| T13 | O/D 지역명 | 최종 기준 O측 8건, D측 13건 수준의 지역명 결측 확인 | `99` 미확인 코드 성격으로 유지 | 공급 데이터의 미확인 지역 의미 보존 |
| T14 | O/D 시군구명 | 일부 지역명 결측 | 최종 파일에 기록, 필요 시 코드 기준 보조 | 핵심 CNT/교통수단/성연령 컬럼은 결측 없음 |
| T25 | O/D 시군구명 | O측 4건, D측 11건 수준의 지역명 결측 확인 | `99` 또는 unknown 성격으로 유지 | 임의 삭제 시 이동량 왜곡 가능 |
| T21 | `CNT` 없음 | 시간대/날짜 보조 테이블 성격 | 모델 후보에서 제외, 참고용 유지 | 유동량 직접 집계 변수 없음 |
| T15/T17/T18/T19 | 파일 없음 | 현재 `data` 폴더에 최종 산출물 없음 | 누락 테이블로 표시 | 팀 공유 시 범위 혼선 방지 |

핵심 분석 컬럼인 `CNT`, `DURATION`, `PURPOSE`, `TRANS_GB`, `SEX_CD`, `AGE_GRP`는 최종 후보 파일 기준 결측이 없거나 분석 가능한 상태로 확인했다. 단, 코드값의 의미는 실제 이동 목적/수단이 아니라 통신사가 추정한 값이므로 보조지표로만 사용한다.


### 6. 코드값 정의 및 해석 기준

`PURPOSE`와 `TRANS_GB`는 `my_eda.ipynb`에서 성남시 제공 코드 정의를 반영했다. 다만 통신사가 위치 반경과 이동 패턴으로 추정한 값이므로 실제 목적/실제 이동수단으로 단정하지 않는다.

#### PURPOSE 코드
| 코드 | 의미 | 활용 판단 |
|---:|---|---|
| 0 | 귀가 | 보조지표 |
| 1 | 출근 | 보조지표 |
| 2 | 등교 | 보조지표 |
| 3 | 쇼핑 | 상권/방문 성격 보조지표 |
| 4 | 관광 | 방문/외부수요 보조지표 |
| 5 | 병원 | 생활서비스 방문 보조지표 |
| 6 | 기타 | 기타 목적 |

#### TRANS_GB 코드
| 코드 | 의미 | 활용 판단 |
|---:|---|---|
| 0 | 차량 | 접근성/교통 보조지표 |
| 1 | 노선버스 | 대중교통 보조지표 |
| 2 | 지하철 | 대중교통 보조지표 |
| 3 | 도보 | 보행 접근성 보조지표 |
| 4 | 고속버스 | 광역 이동 보조지표 |
| 5 | 기차 | 광역 이동 보조지표 |
| 6 | 항공 | 광역 이동 보조지표 |
| 7 | 기타 | 기타 이동수단 |

#### 추가 확인 필요 코드
| 컬럼 | 관측/사용 내용 | 주의점 |
|---|---|---|
| `SEX_CD` | `M`, `F`, `W` 값 관측 | `W`의 정확한 의미는 코드북 확인 필요 |
| `AGE_GRP` | 1~12 범위 값 관측 | 연령대 매핑표 확인 후 해석 필요 |


### 7. EDA 정리

#### 월별 EDA에서 만든 주요 지표
| 원천 | 지표 | 의미 | 사용 방향 |
|---|---|---|---|
| T24 | `floating_pop` | 행정동별 전체 유동인구 규모 | 기본 규모 변수 |
| T24 | `floating_change_rate` | 유동인구 전월 대비 변화율 | 변화 감지 |
| T24 | `floating_pop_per_1k_pop` | 등록 인구 1,000명당 유동인구 | 행정동 간 규모 보정 |
| T24 | `working_pop` | 경제활동 연령층 유동인구 proxy | 소비/활동 인구 보조 |
| T24 | `working_pop_share` | 전체 유동 중 경제활동 연령층 비중 | 지역 이용자 구성 |
| T13 | `external_inflow` | 성남시 외부에서 해당 행정동으로 들어온 이동량 | 외부 수요/압력 후보 |
| T13 | `external_inflow_change_rate` | 외부유입 전월 대비 변화율 | 외부 유입 변화 감지 |
| 인구 | `TOTAL_POP`, `HOUSEHOLDS` | 등록 인구/세대수 | 통신 지표 규모 보정 및 결합 |

#### 분기별 EDA에서 남긴 판단
- 팀 공통 마스터 테이블이 분기 단위라면 통신 데이터도 `행정동-분기` 단위로 집계하는 것이 안정적이다.
- 1차 추천 조합은 `T24 + T13 + T26`이다.
- `T27`은 T26과 중복성을 확인한 뒤 선택적으로 추가한다.
- `T25`는 시군구 OD라 행정동 모델에서는 해상도가 낮아 보조 변수로 둔다.
- `PURPOSE`, `TRANS_GB` 기반 변수는 추정 목적/추정 이동수단이므로 모델의 보조 설명변수로만 사용한다.


### 7-1. 통계 검증 상태

현재 정리본에서 검증된 것과 아직 검증되지 않은 것을 분리한다. 여기서 말하는 검증은 두 종류가 다르다.

| 구분 | 현재 상태 | 의미 |
|---|---|---|
| 데이터 품질 검증 | 완료 | 파일 존재, 행 수, 컬럼 수, 기간 범위, 핵심 컬럼 결측 여부 확인 |
| 전처리 검증 | 완료 | 최종 산출물 재오픈, 지역명 결측/99 코드 처리 기준 문서화 |
| EDA 수준 검토 | 일부 완료 | 월별/분기별 추세, 상관, 변동성, 후보 변수 방향성 확인 |
| 통계적 유의성 검정 | 일부만 완료 | 기존 `eda.ipynb`에서 일부 지표의 변동성, 지역 차이, 상관, 추세를 점검했지만 전체 T4~T27 변수에 대해 모두 검정한 것은 아님 |
| 기존 EDA 통계 점검 | 일부 완료 | 변동성 요약, 지역 차이 검정, 상관계수, 월별 방향성 확인을 수행했으나 최종 feature table 검정은 별도 필요 |
| 모델링 검증 | 미완료 | 최종 목표변수와 마스터 테이블이 확정된 뒤 train/test, 변수 중요도, 성능 검증 필요 |

#### 현재 문서에서 조심해야 할 표현
- `핵심 후보`, `보조 후보`는 통계적으로 유의하다는 뜻이 아니라 **EDA와 데이터 구조상 먼저 써볼 후보**라는 뜻이다.
- `PURPOSE`, `TRANS_GB`는 코드 정의는 확인했지만 통신 추정값이므로 실제 목적/수단으로 단정하지 않는다.
- 최종 보고서에는 “통계적으로 검증됨”이 아니라 “EDA 기반 후보로 선정했고, 최종 모델링 단계에서 검증 예정”이라고 쓰는 것이 안전하다.

#### 추가로 돌려야 할 통계 검증
| 검증 | 목적 | 적용 대상 |
|---|---|---|
| 기술통계/분포 | 변수의 결측, 왜도, 극단값 확인 | 모든 최종 feature |
| 지역 차이 검정 | 구/행정동별 차이가 우연인지 확인 | T24 유동량, T13 외부유입, T26 체류시간 |
| 상관/중복성 확인 | 비슷한 변수가 너무 많이 들어가는지 확인 | 최종 feature 전체 |
| 추세 검정 | 시간에 따라 증가/감소 경향이 있는지 확인 | 월별/분기별 feature |
| 목표변수 연관성 | 젠트리피케이션 proxy/target과 실제 관련 있는지 확인 | 최종 모델링 테이블 |


### 7-2. 공시지가 EDA 정리

통신 T데이터 외에 `공시지가_test.ipynb`에서 공시지가 상승률 EDA도 진행했다. 이 내용은 모델링에서 지역 토지가치 변화 보조지표로 연결할 수 있다.

| 항목 | 내용 |
|---|---|
| 입력 파일 | `data/2차전처리_공시지가데이터.csv` |
| 작업 노트북 | `Jiryun/공시지가_test.ipynb` |
| 데이터 규모 | 15,542행, 13컬럼 |
| 기간 | 2023~2025년, 1월 기준 중심. 7월 자료는 소수 존재 |
| 공간 단위 | 법정동/구, 필지 고유번호 기준 |
| 주요 컬럼 | `고유번호`, `법정동코드`, `법정동명`, `구`, `기준연도`, `기준월`, `공시지가` |

#### 계산한 지표
| 지표 | 계산 방식 | 의미 |
|---|---|---|
| `공시지가_2023`, `공시지가_2024`, `공시지가_2025` | 같은 `고유번호`의 1월 공시지가를 연도별 wide 형태로 변환 | 연도별 가격 수준 |
| `상승률_23_24` | `공시지가_2024 / 공시지가_2023 - 1` | 2023~2024 상승률 |
| `상승률_24_25` | `공시지가_2025 / 공시지가_2024 - 1` | 2024~2025 상승률 |
| `상승률_23_25` | `공시지가_2025 / 공시지가_2023 - 1` | 2023~2025 누적 상승률 |
| `연평균상승률_23_25` | `(공시지가_2025 / 공시지가_2023) ** (1/2) - 1` | 2년 연평균 상승률 |

#### EDA 결과 메모
- 1월 기준 전체 고유번호 5,178개 중 2023~2025가 모두 존재하는 필지는 5,176개다.
- 법정동은 43개이며, `필지수 >= 20` 조건을 적용하면 40개 법정동이 비교 대상이다.
- 2023~2025 누적 상승률 중앙값 상위는 수정구 시흥동, 분당구 삼평동, 분당구 백현동, 수정구 금토동, 분당구 서현동 순으로 확인했다.
- 하위는 수정구 상적동, 중원구 상대원동, 중원구 금광동, 중원구 여수동, 중원구 도촌동 순으로 확인했다.
- 법정동 대표값은 평균보다 중앙값을 우선 사용했다. 일부 필지의 극단값이 평균을 크게 흔들 수 있기 때문이다.
- 공시지가 상승률은 분기별 통신 지표처럼 직접 해석하기보다, 연도별 지역 토지가치 변화 보조지표로 쓰는 것이 안전하다.
- 통신 데이터는 행정동 기준이 많고 공시지가는 법정동 기준이므로, 최종 결합 전 법정동-행정동 매핑 기준을 따로 검증해야 한다.


### 7-3. 보조 작업 간단 메모

아래 내용은 주요 결론이 아니라 기존 노트북을 다시 보며 빠뜨리지 않도록 간단히 남긴 보조 메모다.

| 항목 | 간단 메모 |
|---|---|
| T20 | `DISTANCE`, `CARBON_EMISSIONS`가 있어 거리/탄소 참고 변수로만 기록한다. |
| T21 | `CNT`가 없으므로 직접 feature 후보보다는 시간대 참고용으로 둔다. |
| T11 | 작업 중 중복 확인 과정이 있었지만 최종 파일 기준으로 정리되었으므로 중요 이슈로 따로 해석하지 않는다. |
| T22 | 생산가능/소비가능 연령층 비중은 보조 후보로만 기록한다. |
| 통계 점검 | 변동성, 지역 차이, 상관, 월별 방향성은 일부 EDA에서 봤지만 전체 feature 검증은 아직 미완료다. |


### 8. 모델링용 테이블 우선순위

| 우선순위 | 테이블 | 사용 판단 | 이유 |
|---:|---|---|---|
| 1 | T24 | 기본 feature 후보 | 행정동 단위 유동인구라 `base_quarter + ADMI_CD`로 바로 집계하기 좋음 |
| 2 | T13 | 핵심 추가 후보 | 행정동 OD라 외부유입, 유출입 구조, 이동량 증가를 만들기 좋음 |
| 3 | T26 | 핵심 추가 후보 | 도착 행정동 기준 체류시간과 목적/수단 특성을 만들기 좋음 |
| 4 | T27 | 선택 추가 후보 | 출발 행정동 기준 이동수단/목적/체류 특성을 보완하되 T26과 중복 확인 필요 |
| 보조 | T25 | 참고 후보 | 시군구 OD라 행정동 모델에서는 해상도가 낮아 보조 지표에 적합 |
| 보조 | T22 | 참고 후보 | 행정동 성·연령·내외국인 구성 보조 지표 가능 |
| 보류 | T4~T12, T14, T16, T20, T21, T23 | 구조 파악/필요 시 추가 | 핵심 후보와 중복되거나 모델 단위와 맞추기 전 추가 검토 필요 |

1차 모델링은 `T24 + T13 + T26`으로 시작하고, 변수 중요도와 중복성 확인 후 `T27`, `T25`, `T22`를 추가 검토한다.


### 9. 최종 변수 연결표

| 원천 | 파생변수 후보 | 집계 기준 | 의미 | 사용 방향 |
|---|---|---|---|---|
| T24 | `q_floating_pop_sum` | `base_quarter + ADMI_CD` | 분기별 행정동 전체 유동량 | 기본 규모 변수 |
| T24 | `q_floating_pop_per_1k_pop` | `base_quarter + ADMI_CD` | 인구 1,000명당 유동량 | 규모 보정 변수 |
| T24 | `q_purpose_*_ratio` | 목적별 CNT / 전체 CNT | 목적별 유동 비중 | 지역 성격 비교 |
| T24 | `q_night_ratio` | 야간 CNT / 전체 CNT | 야간 유동 비중 | 주거/상권 혼합 특성 |
| T24 | `q_lunch_ratio`, `q_evening_ratio` | 시간대별 CNT / 전체 CNT | 점심/저녁 활동 비중 | 상권 시간대 특성 |
| T24 | `q_age_20_40_ratio` | 연령대 CNT / 전체 CNT | 주요 소비/경제활동 연령층 비중 | 소비층 proxy |
| T13 | `q_inflow_cnt` | 도착 행정동 기준 | 해당 행정동으로 들어온 이동량 | 유입 규모 |
| T13 | `q_outflow_cnt` | 출발 행정동 기준 | 해당 행정동에서 나간 이동량 | 유출 규모 |
| T13 | `q_external_inflow_cnt` | 외부 지역 -> 해당 행정동 | 외부유입량 | 외부 수요 증가 proxy |
| T13 | `q_external_inflow_ratio` | 외부유입 / 전체유입 | 외부유입 비중 | 젠트리피케이션 압력 후보 |
| T26 | `q_avg_stay_time` | 도착 행정동 기준 | 평균 체류시간 | 체류형 상권 여부 |
| T26 | `q_long_stay_ratio` | 장기체류 CNT / 전체 CNT | 장기체류 비중 | 체류 강도 |
| T26 | `q_purpose_stay_avg_*` | PURPOSE별 DURATION 평균 | 목적별 체류시간 | 방문 목적별 체류 특성 |
| T27 | `q_transport_*_ratio` | TRANS_GB별 CNT 비중 | 이동수단별 비중 | 접근성/교통 특성 |
| T27 | `q_purpose_transport_*` | PURPOSE + TRANS_GB | 목적·수단 조합 | 접근 방식과 방문 목적 결합 |
| T25 | `q_city_inflow_pressure` | 시군구 OD 기준 | 구 단위 유입 압력 | 행정동 모델 보조 변수 |
| T22 | `q_age_sex_foreigner_mix` | 행정동 + 시간대 | 성·연령·내외국인 구성 | 인구 구성 보조 변수 |
| T22 | `working_age_share`, `consumer_age_share` | 행정동 + 시간대 | 생산가능/소비가능 연령층 비중 | 인구 구성 보조 변수 |
| T20 | `distance_sum`, `distance_mean` | 시군구 + 월/요일 | 이동 거리 규모/평균 | 접근성 보조 변수 |
| T20 | `carbon_emissions_sum`, `carbon_emissions_mean` | 시군구 + 월/요일 | 탄소배출량 규모/평균 | 이동수단/거리 파생 보조 |
| 공시지가 | `official_land_price_2023/2024/2025` | 고유번호 + 기준연도 | 연도별 필지 공시지가 | 가격 수준 보조 |
| 공시지가 | `official_land_price_growth_23_25` | 고유번호 | 2023~2025 누적 상승률 | 토지가치 상승 속도 |
| 공시지가 | `dong_land_price_growth_median` | 법정동 | 법정동 상승률 중앙값 | 지역 단위 보조 feature |

#### 공통 원본 컬럼 연결
| 원본 컬럼 | 표준 의미 | 사용 방향 |
|---|---|---|
| `ETL_YM`, `ETL_YMD` | 기준 월/일 | 월별 또는 분기별 기준 생성 |
| `CNT` | 집계 건수/유동량 | 대부분의 핵심 feature 원천 |
| `DURATION` | 체류시간 | T16/T26/T27 체류 변수 원천 |
| `PURPOSE` | 추정 이동 목적 코드 | 목적별 비중/목적별 체류 보조 |
| `TRANS_GB` | 추정 이동수단 코드 | 교통수단별 비중 보조 |
| `SEX_CD`, `AGE_GRP` | 성별/연령대 코드 | 이용자 구성 보조 |
| `TIME_CD`, `D_TIME_CD`, `O_TIME_CD` | 시간대 코드 | 야간/점심/저녁 시간대 지표 |
| `ADMI_CD`, `D_ADMI_CD`, `O_ADMI_CD` | 행정동 코드 | 행정동 단위 집계 키 |
| `CTY_NM`, `ADMI_NM`, `D_*`, `O_*` | 지역명 | 해석/시각화 라벨 |


### 10. 최종 산출물 요약표

| 산출물 | 파일/노트북 | 내용 | 팀 공유 시 사용 |
|---|---|---|---|
| T데이터 최종 파일 목록 | `data/t*_2023_2025_all_*` | T4~T27 중 확보된 통신 데이터 최종 parquet/csv | 데이터 목록 공유 |
| 전처리 기록 | `1차_전처리_small.ipynb`, `1차_전처리.ipynb` | 날짜, 결측, 지역 매핑, 코드값, 저장 검증 | 처리 근거 확인 |
| 월별 EDA | `eda.ipynb`, `my_eda_월별EDA_백업.ipynb` | 유동인구/외부유입/인구 결합/월별 지표 해석 | 월별 변수 후보 확인 |
| 분기별 EDA | `my_eda.ipynb` | 행정동-분기 패널, T24/T13/T26 중심 변수 후보 | 모델링 단위 확정 전 검토 |
| 공시지가 EDA | `공시지가_test.ipynb` | 필지별/법정동별/구별 공시지가 상승률, 시각화, 해석 메모 | 토지가치 변화 보조지표 |
| 최종 정리본 | 현재 노트북 | 전체 T데이터 파일, 공시지가 EDA, 전처리 기준, EDA 판단, 변수 연결표 통합 | 튜터님/팀 공유용 |


### 11. 최종 체크리스트

- [x] `T4~T27` 중 현재 확보된 최종 산출물 목록 정리
- [x] `T15/T17/T18/T19` 현재 폴더 기준 누락 표시
- [x] 테이블별 행 수, 컬럼 수, 기간 범위 정리
- [x] 원본/중간 파일 -> 최종 산출물 매핑표 보완
- [x] 통합/전처리 기준 문서화
- [x] 핵심 컬럼 결측 검증 코드 정리
- [x] 문제 데이터 및 예외 케이스 정리
- [x] `PURPOSE`, `TRANS_GB` 코드 정의와 해석 주의점 정리
- [x] 월별/분기별 EDA 결과 기반 변수 후보 정리
- [x] 통계 검증 완료/미완료 상태 구분
- [x] 공시지가 EDA 정리 포함 완료
- [x] 기존 노트북 재대조 후 세부 누락 보완 완료
- [x] 모델링용 테이블 우선순위 정리
- [x] 최종 변수 연결표 작성

### 12. 팀 공유용 5줄 요약

1. 통신 T데이터는 현재 `T4~T27` 중 확보된 20개 테이블을 2023~2025 최종 parquet/csv 산출물 기준으로 정리했고, 공시지가 상승률 EDA도 함께 포함했다.
2. 전처리에서는 날짜 범위, 지역 매핑, 핵심 컬럼 결측, 코드값 분포를 확인했고, 공급 데이터의 미확인 코드로 보이는 `99`는 원본 의미 보존을 위해 유지했다.
3. 월별 EDA에서는 T24 유동인구와 T13 외부유입을 중심으로 행정동별 변화율과 인구 보정 지표를 만들었다.
4. 분기별 모델링 후보는 `T24 + T13 + T26`을 1차 조합으로 보되, 아직 통계적 유의성이 확정된 것은 아니다.
5. 최종 목표변수와 마스터 테이블이 확정되면 지역 차이, 상관/중복성, 시간 추세, 모델 성능, 법정동-행정동 매핑 검증을 추가로 진행해야 한다.


**취합 메모**: `Jiryun/개인_통합정리_지륜템플릿_작성본.ipynb`에서 마크다운 셀 17개를 취합했다. 실행 코드와 산출물 생성 로직은 원본 노트북 및 아래 최종 분석 템플릿 코드 셀을 함께 참조한다.


## 개인별 정리: sungju / 개인_통합정리_성주.ipynb



## 성주 개인 통합 정리 Notebook

#### 작성 메타
- 프로젝트명: 성남시 젠트리피케이션 위험도 분석 시스템 구축 및 상생 체계 제안
- 담당자: 성주
- 담당 파트: 교통(지하철·버스), 신용(대민개방·전입·전출), 공시지가
- 작성일: 2026-04-29
- 목적: 각 데이터의 1차/2차 전처리 결과와 예외 처리 기준을 한 노트북에 통합 정리하여 팀 공유 가능한 재현 가능한 형태로 만든다.
- 최종 산출물: `2차전처리_지하철데이터.csv`, `2차전처리_버스데이터.csv`, `2차전처리_대민개방데이터.csv`, `2차전처리_전입데이터.csv`, `2차전처리_전출데이터.csv`, `2차전처리_공시지가데이터.csv` + 예외 처리 기준표

> 핵심 산출물은 "예외처리 기준의 명문화"이다. 특히
> - 버스 정류소-행정동 보정표,
> - 산 필지 제외 기준,
> - 신용 데이터 0값(triple_zero) 분류 기준
> 은 표로 같이 남겨야 팀이 재현할 수 있다.


### 1. 작업 개요

- 내가 맡은 데이터: 교통(지하철/버스), 신용(대민개방·전입·전출), 공시지가
- 왜 필요한지: 분석 결과보다 "어떤 값을 어떻게 처리했는지"가 재현성에 직접 영향을 주기 때문. 특히 0값과 행정동 매핑은 모든 조인 단계에서 영향을 준다.
- 최종적으로 남길 표/파일:
  - 문제 데이터 발견 목록 표
  - 정류소-행정동 보정표 (구체적 정류소ID 포함)
  - 산/공원/비주거성 필지 제외 기준
  - 신용 데이터 0값 분류 기준(structural_zero / error_zero)
  - clean 전후 비교 결과
  - 교통 EDA 핵심 해석 요약


### 2. 파일 정리 + 입력 파일 목록

#### 작업 파일 정리표
| 파일명 | 현재 역할 | 유지 여부 | 비고 |
|---|---|---|---|
| (삭제 예정)1차 전처리.ipynb | 초기 작업본 | 삭제 예정 | 백업만 보관 |
| 2차 전처리_교통.ipynb | 교통(지하철/버스) 처리 | 유지 | 정류소-행정동 보정표 포함 |
| 2차 전처리_신용.ipynb | 신용/전입/전출 처리 | 유지 | triple_zero 분류 기준 포함 |
| 2차 전처리_공시지가.ipynb | 공시지가 처리 | 유지 | 산 필지 제외 |
| EDA_교통.ipynb | 교통 EDA | 유지 | HHI / 로그 변화율 / 사분면 분석 결과 |


#### 입력 원본 파일 목록 (사용한 실제 파일)
| 파일명 | 설명 | 사용 여부 | 비고 |
|---|---|---|---|
| ../Data/subway.csv | 지하철 월별 승하차 | 사용 | 결측 0건 |
| ../Data/bus_plz.csv | 버스 정류소 일별 승하차 | 사용 | 행정동 65,110건 결측 → 매핑 보정 후 0건 |
| ../Data/신용정보.csv | 행정동/연령별 신용 인구 통계 | 사용 | 504행 triple_zero 검출 |
| ../Data/전입통계.csv | 전입 통계 | 사용 | 결측 0건 |
| ../Data/전출통계.csv | 전출 통계 | 사용 | 결측 0건 |
| ../Data/성남시_공시지가_통합__202604221844.csv | 공시지가 | 사용 | 산 필지 15행 → 제외 |
| ../Data/202301_202306_연령별인구현황_월간.csv | 주민등록 인구 (검증용) | 검증용 | 신용 0값 검증에 사용 |


### 3. 데이터 로드

각 파일을 동일한 인코딩(`utf-8-sig` 또는 `cp949`) 규칙으로 불러온다.


### 4. 데이터 기본 점검

각 데이터셋의 행/열, 결측치, 중복, 타입을 한 번에 확인한다.


### 5. 문제 데이터 발견 목록 / 처리 기준 문서화

#### 5-1. 문제 데이터 발견 목록 표

| 데이터셋 | 컬럼 | 문제 유형 | 발견 내용 | 처리 방식 | 이유 |
|---|---|---|---|---|---|
| 버스 | 행정동 | 결측 | 65,110건 행정동 NaN | 정류소ID 또는 정류소명 기반 매핑 보정 | 공간 키 누락 시 조인 불가 |
| 버스 | 행정동 | 중복 매핑 | 정류소ID 206000616, 206000617 — 동일 정류소가 백현동/삼평동 양쪽에 매핑 | 잘못된 행을 삭제 | 동별 합계가 부풀려짐 |
| 버스 | 정류소명 | 변경 | 2023년 ↔ 2024년 정류소명 변경 (성남역 개통 영향) | 통합 명칭으로 재할당 | 동일 위치 식별 |
| 버스 | 데이터 범위 | 성남 외 | 고기3리.유원지입구 등 용인 정류소 포함 | 행 삭제 | 분석 범위 외 |
| 신용 | TOT_CNT/ECON_CNT/NECON_CNT | 0값 (triple_zero) | 504행 / 운중동·고등동 집중 | structural_zero / error_zero 분류 후 분석에서 제외 (원본은 보존) | 임의 평균 대체 시 왜곡 |
| 공시지가 | 특수지구분명 | 산 필지 | 5필지 × 3년 = 15행 | 분석에서 제외 | 주거/상업 분석 대상 외 |
| 공시지가 | 공시지가 | 이상치 후보 | IQR 기준 1,168행 상위 이탈 | 제거하지 않고 boxplot/log 변환으로 점검만 수행 | 실제 강남급 도심 가격대 가능 |
| 지하철 | 전 컬럼 | 결측 | 0건 | 그대로 유지 | 처리 불필요 |
| 전입/전출 | 전 컬럼 | 결측 | 0건 | 그대로 유지 | 처리 불필요 |

#### 5-2. 0값 처리 기준표 (신용 데이터)

| 분류 | 정의 | 처리 방식 |
|---|---|---|
| `normal` | 그룹 내에 0이 아닌 값이 함께 존재 | 그대로 사용 |
| `error_zero` | 같은 (월·BCD·연령·동) 그룹 안에 0과 비0이 공존 | 분석에서 제외 + 감사 로그 보관 |
| `structural_zero` | 그룹 전체가 모두 0 | 분석에서 제외 + 별도 점검 후 보고 |

#### 5-3. 원본 유지 / 분석용 NaN 처리 원칙

- 원본 유지 원칙: 모든 raw csv는 절대 덮어쓰지 않는다. 분석용 사본(`clean_df`)에서만 처리한다.
- 분석용 처리 원칙: 임의 평균/중앙값 대체보다 `exclude_from_analysis` 플래그 또는 행 제거를 우선한다.
- 예외: 행정동 매핑은 명문화된 보정표에 한해 직접 치환한다.


### 6. 성주 전용 기준표

#### 6-1. 정류소-행정동 보정표 (실제 적용)

| 정류소 ID | 정류소번호 | 연도 / 정류소명 | 기존 행정동 | 수정 행정동 | 수정 사유 |
|---|---|---|---|---|---|
| 206000305 | 7159 | 2023: 아름마을.이매고교.효성선경아파트.하나은행<br>2024: 성남역.이매고교.아름마을.효성선경아파트 | NaN | 이매2동 | 정류소명 변경 + 동일 위치 확인 |
| 206000314 | 7303 | 2023: 아름마을.이매고교.효성선경아파트.하나은행<br>2024: 성남역.이매고교.아름마을.효성선경아파트 | NaN | 이매2동 | 정류소명 변경 + 동일 위치 확인 |
| 206000530 | 7487 | 2023: 백현마을2단지<br>2024: 성남역.백현마을2단지 | NaN | 백현동 | 정류소명 변경 + 동일 위치 확인 |
| 206000537 | 7420 | 2023: 백현마을3단지<br>2024: 성남역.백현마을3단지 | NaN | 백현동 | 정류소명 변경 + 동일 위치 확인 |
| 206000616 | 7556 | 2023: 보평중고등학교<br>2024: 성남역.보평중고등학교 | 백현동, 삼평동 | 삼평동 유지 / 백현동 삭제 | 중복 매핑 제거 |
| 206000617 | 7557 | 2023: 보평중고등학교<br>2024: 성남역.보평중고등학교 | 백현동, 삼평동 | 백현동 유지 / 삼평동 삭제 | 중복 매핑 제거 |

> 그 외 정류소명 → 행정동 매핑 dictionary 4세트 (`dong_map`, `dong_map_update`, `dong_map_update2`, `dong_map_update3`, `dong_map_update4`) 적용. 모든 NaN 채움 후 잔여 결측은 0건.

#### 6-2. 성남시 외 정류소 제외
- 고기3리.유원지입구 / 고기동마을 / 고기초등학교 / 왕재건설중기 → 행 삭제 (용인시 정류소)

#### 6-3. 산 / 공원 / 비주거성 필지 제외 기준 (공시지가)

| 분류 | 위치 | 판단 | 처리 |
|---|---|---|---|
| 산 | 분당구 동원동 67-8 | 산지 | 삭제 |
| 산 | 수정구 창곡동 116-4 | 산지 | 삭제 |
| 공원 | 수정구 수진동 41-3 | 공원 | 삭제 |
| 산 안 시설 | 수정구 상적동 52-3 | 관리사무소 추정 | 삭제 |
| 임야상 주택 | 수정구 단대동 164-3 | 토지계획상 임야 | 삭제 |

> 분석 대상과 거리가 멀어 모두 삭제 처리.

#### 6-4. 산번지 예외 검토 조건
- 실제 상업/주거 활용 흔적이 확인되는 경우
- 팀 분석 범위상 포함 근거가 분명한 경우
- 제외 시 표본 손실이 매우 크고 별도 표기 후 활용 가능한 경우


### 7. 전처리 실행 — 지하철

지하철 데이터는 결측·중복 모두 0이라 별도 처리 없이 저장한다.


### 8. 전처리 실행 — 버스

1) 정류소ID 단위 중복 매핑 제거 → 2) 정류소명 통일 → 3) 행정동 매핑 보정 → 4) 성남 외 정류소 삭제 → 5) 중복 제거 → 6) 월 단위 집계


### 9. 전처리 실행 — 신용 (대민개방)

`TOT_CNT == 0 & ECON_CNT == 0 & NECON_CNT == 0` 인 504행을 **structural_zero / error_zero** 로 분류하고, 분석용 사본에서만 제외한다. 원본은 보존.

추가로 2023년 상반기 구간은 `202301_202306_연령별인구현황_월간.csv` 로 검증 → 운중동의 0값 대부분이 `주민등록상 인구 있음 → 신용 0 재점검 필요` 로 판정됨.


#### 9-1. 주민등록 데이터로 신용 0값 검증 (2023년 상반기)


### 10. 전처리 실행 — 전입 / 전출

전입·전출 통계는 결측·이상치 점검 결과 모두 0이라 그대로 저장한다.


### 11. 전처리 실행 — 공시지가

1) 산 필지 5건(× 3년 = 15행) 제외
2) 법정동명에서 구(분당/수정/중원) 추출
3) 기준연도 2023~2025, 기준월 1·7월(연 2회) 확인
4) IQR / boxplot / log 변환으로 이상치 점검 (제거하지 않음)


### 12. clean 파일 저장 전후 비교

각 데이터셋의 처리 전/후 행 수 변화와 처리 원칙을 정리한다.

| 데이터셋 | before shape | after shape | 변경 내용 |
|---|---|---|---|
| subway | (850, 5) | (850, 5) | 변경 없음 (결측 0건) |
| bus | (2,058,750, 10) | 약 (1,297,105, 10) | 정류소-행정동 보정 + 중복 제거 + 월 집계 |
| credit | (29,818, 9) | 약 (29,332, 12) | triple_zero 504행 분석 제외 + 컬럼 3개 추가 |
| in(전입) | (405,746, 31) | (405,746, 31) | 변경 없음 |
| out(전출) | (402,335, 31) | (402,335, 31) | 변경 없음 |
| price(공시지가) | (15,557, 11) | (15,542, 12) | 산 필지 15행 제외 + 구 컬럼 추가 |

#### 최종 처리 원칙 요약

- **0값**은 실제 0인지 결측 대체값인지 먼저 확인하고, 원본은 유지한 채 분석용 사본에서만 별도 처리한다.
- **정류소-행정동 불일치**는 보정표를 별도로 작성하고, 근거 없는 일괄 치환은 하지 않는다.
- **공시지가의 산 필지**는 분석 대상 외이므로 제외하되, 기준은 "특수지구분명 == '산'" 으로 명문화한다.
- **신용 데이터의 triple_zero**는 임의 대체보다 `exclude_from_analysis` 플래그 + 감사 로그 보관 원칙을 우선한다.
- **이상치(공시지가)**는 IQR 기준만으로 제거하지 않는다. 도심 고가 필지를 정상값으로 인정하기 위함.


### 13. 교통 EDA 핵심 결과 요약

`EDA_교통.ipynb` 에서 도출한 핵심 해석을 한 페이지로 정리.

#### 13-1. 시계열 패턴
- **2024년 10월 (≒ 2024년 3분기)** 이 행정동·역별 이용량의 **구조적 변화 시점**으로 관측됨 → 정책/노선 변경/외부 이벤트 가능성.
- 행정동 간 격차는 계속 유지(상위는 계속 상위, 하위는 계속 하위), 전체 수요는 증가 추세.

#### 13-2. 로그 변화율 분석 (2023-01 vs 최종월 / 분기)
- 양수(파랑) = 외부 소비자 유입 증가 → **임대료 상승 압력 높은 지역**
- 음수(빨강) = 소비자 이탈 → **소상공인 이탈 가속 가능성**
- 1,000 미만 소규모 행정동은 통계적 왜곡 방지를 위해 제외.

#### 13-3. HHI(허핀달-허시만) 기반 정류소 집중도
- HHI 높음 → 대형 허브에 집중 → 골목 상권 집객력 약화 → 폐업 압력
- HHI 낮음 → 다수 정류소 분산 → 골목 상권/전통시장 살아있음

#### 13-4. 산점도 — 소비 유입 × 집중도 사분면

| 사분면 | 이용량 | HHI | 해석 |
|---|---|---|---|
| 우상단 | 증가 | 높음 | **위험 의심** — 소비자 늘었지만 대형 허브에만 몰림 |
| 우하단 | 증가 | 낮음 | 골목 상권 전반 활성화 — 양극화 없음 |
| 좌상단 | 감소 | 높음 | 소규모 상권 붕괴 완료 — 대형 허브만 잔존 |
| 좌하단 | 감소 | 낮음 | 상권 전반 쇠퇴 — 소비자 자체 감소 |

#### 13-5. 정류소명 키워드 기반 상업형 / 생활형 분류
- 상업형 키워드: 역, 터미널, 백화점, 플라자, 몰, 아울렛, 마트, 쇼핑, 시장
- 생활형 키워드: 아파트, 학교, 병원, 행정복지센터 등
- **상업형 비중 높은 동** → 외부 소비 유입 구조 → 임대료 상승·소상공인 이탈 압력 높음
- **생활형 비중 높은 동** → 골목 상권 중심, 상업화 압력 낮음

#### 13-6. 지하철 핵심
- 야탑·신흥·가천대: 하차 우세 → 외부 유입 중심 → 상업 기능 강화 가능성
- 수내·서현·정자: 승차 우세 → 주거/통근 중심 → 소비 유출 구조
- 경강선/신분당선 HHI 높음(집중형), 8호선/분당선 HHI 낮음(분산형)
- 우상단(하차 증가 + 하차비율 > 0.5) 역 = 젠트리피케이션 압력 가장 높은 후보


### 14. 최종 체크리스트

- [x] 문제 데이터 발견 목록 표 작성
- [x] 정류소-행정동 보정표 작성 (정류소ID 단위)
- [x] 산 / 공원 / 비주거성 필지 제외 기준 작성
- [x] 산번지 예외 검토 조건 정리
- [x] 신용 0값 분류 기준(structural_zero / error_zero) 작성
- [x] 주민등록 데이터 기반 신용 0값 교차 검증
- [x] 원본 유지 / 분석용 NaN 처리 원칙 문서화
- [x] clean 파일 저장 전후 비교
- [x] 교통 EDA 핵심 결과 요약 (시계열·HHI·사분면·상업/생활 분류)
- [x] 6개 clean csv 저장 완료


### 15. 팀 공유용 5줄 요약

1. 교통(지하철·버스), 신용(대민·전입·전출), 공시지가 6종 데이터의 1차/2차 전처리 결과를 정리하고 예외 처리 기준을 모두 표로 명문화했다.
2. 버스 데이터에서 행정동 결측 65,110건과 정류소ID 단위 중복 매핑(206000616/617)을 보정표 기반으로 처리해 잔여 결측 0건을 달성했다.
3. 신용 데이터의 504행 triple_zero는 `structural_zero / error_zero` 로 분류해 분석에서 제외했고, 운중동 0값은 주민등록 인구와 매칭하여 "신용 0 재점검 필요"로 판정했다.
4. 공시지가에서는 산 필지 15행만 제외하고 IQR 이상치는 도심 고가 필지를 보존하기 위해 제거하지 않았다.
5. 교통 EDA에서 2024년 3·4분기를 구조적 변화 시점으로 식별하고, HHI×하차 로그 변화율 사분면으로 젠트리피케이션 위험 지역 후보를 분류할 수 있는 분석 틀을 정리했다.


**취합 메모**: `sungju/개인_통합정리_성주.ipynb`에서 마크다운 셀 17개를 취합했다. 실행 코드와 산출물 생성 로직은 원본 노트북 및 아래 최종 분석 템플릿 코드 셀을 함께 참조한다.


## 6. 공통 키 점검 및 통일

### 왜 공통 키가 필요한가요?
여러 사람이 전처리한 데이터를 하나로 합치려면 같은 지역과 같은 시점을 정확히 가리키는 기준이 필요합니다. 이 기준이 바로 공통 키입니다. 공통 키가 맞지 않으면 병합 결과가 틀어지고, 행 수가 이상하게 늘거나 값이 비는 문제가 생깁니다.

### 최종적으로 사용할 키를 적는 칸
- 최종 키 구조 후보 1: 기준년월 + 행정동코드
- 최종 키 구조 후보 2: 기준년월 + 블록코드
- 실제 최종 확정 키: ________________________________________________

In [ ]:
all_frames = {
    'df_geunsu_store': df_geunsu_store,
    'df_geunsu_sales': df_geunsu_sales,
    'df_geunsu_landprice': df_geunsu_landprice,
    'df_jiryun_t13': df_jiryun_t13,
    'df_jiryun_t25': df_jiryun_t25,
    'df_jiryun_t26': df_jiryun_t26,
    'df_jiryun_t27': df_jiryun_t27,
    'df_eunbi_deal': df_eunbi_deal,
    'df_eunbi_pop': df_eunbi_pop,
    'df_eunbi_firm': df_eunbi_firm,
    'df_seongju_credit': df_seongju_credit,
    'df_seongju_bus': df_seongju_bus,
    'df_seongju_subway': df_seongju_subway,
    'df_seongju_traffic': df_seongju_traffic,
}

alias_map = {
    '기준년월': ['기준년월', 'base_ym', 'stdr_ym', 'ym', '연월', 'year_month'],
    '행정동코드': ['행정동코드', 'dong_cd', 'adm_cd', 'admdong_cd', '행정동코드값'],
    '법정동코드': ['법정동코드', 'bjd_cd', 'legal_dong_cd'],
    '블록코드': ['블록코드', 'block_cd', 'grid_cd'],
    '행정동명': ['행정동명', 'dong_name', 'adm_nm', '행정동'],
}

def rename_to_standard_keys(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    result = df.copy()
    for std_col, aliases in alias_map.items():
        if std_col in result.columns:
            continue
        for alias in aliases:
            if alias in result.columns:
                result = result.rename(columns={alias: std_col})
                break

    # 기준년월 포맷 통일 예시
    if '기준년월' in result.columns:
        temp = result['기준년월'].astype(str).str.replace('.0', '', regex=False).str.strip()
        result['기준년월'] = pd.to_datetime(temp, errors='coerce')

    # 코드 컬럼은 앞자리 0 보존을 위해 문자열로 통일합니다.
    for code_col in ['행정동코드', '법정동코드', '블록코드']:
        if code_col in result.columns:
            result[code_col] = (
                result[code_col]
                .astype(str)
                .str.replace('.0', '', regex=False)
                .str.strip()
                .replace({'nan': np.nan, 'None': np.nan})
            )

    # 연 단위 데이터만 있는 경우 월 기준으로 확장/매핑이 필요할 수 있습니다.
    # 예: 2023년 데이터만 있다면 2023-01 ~ 2023-12로 확장하거나, 연단위로 별도 병합 규칙을 정해야 합니다.
    # 실제 상황에 맞게 이 부분을 수정하세요.

    return result

for name, df in list(all_frames.items()):
    all_frames[name] = rename_to_standard_keys(df)
    print(f'\n===== {name} =====')
    print('columns:', all_frames[name].columns.tolist())
    existence = {col: (col in all_frames[name].columns) for col in KEY_CANDIDATES}
    print('key existence:', existence)

df_geunsu_store = all_frames['df_geunsu_store']
df_geunsu_sales = all_frames['df_geunsu_sales']
df_geunsu_landprice = all_frames['df_geunsu_landprice']
df_jiryun_t13 = all_frames['df_jiryun_t13']
df_jiryun_t25 = all_frames['df_jiryun_t25']
df_jiryun_t26 = all_frames['df_jiryun_t26']
df_jiryun_t27 = all_frames['df_jiryun_t27']
df_eunbi_deal = all_frames['df_eunbi_deal']
df_eunbi_pop = all_frames['df_eunbi_pop']
df_eunbi_firm = all_frames['df_eunbi_firm']
df_seongju_credit = all_frames['df_seongju_credit']
df_seongju_bus = all_frames['df_seongju_bus']
df_seongju_subway = all_frames['df_seongju_subway']
df_seongju_traffic = all_frames['df_seongju_traffic']

selected_keys = None
non_empty_frames = [df for df in all_frames.values() if not df.empty]
for key_option in PREFERRED_KEY_OPTIONS:
    if non_empty_frames and all(set(key_option).issubset(df.columns) for df in non_empty_frames if not df.empty):
        selected_keys = key_option
        break

if selected_keys is None:
    selected_keys = ['기준년월', '행정동코드']

print('\n[선택된 병합 키]', selected_keys)

# 필요 시 실제 컬럼명이 다르면 여기서 rename 예시를 추가하세요.
# 예: df_geunsu_store = df_geunsu_store.rename(columns={'base_month': '기준년월'})

## 7. 개인별 데이터셋을 최종 변수 중심으로 축약

모든 원본 컬럼을 다 들고 병합하면 너무 복잡해집니다. 따라서 최종 분석에 실제로 사용할 핵심 변수만 남겨 축약 DataFrame을 만듭니다.

### 초보자 안내
- 아래 후보 컬럼명은 예시입니다. 실제 컬럼명이 다르면 `실제 컬럼명 확인 후 수정` 주석이 있는 부분만 바꾸면 됩니다.
- 후보 컬럼이 여러 개면 가장 먼저 발견된 컬럼을 사용하도록 구성했습니다.

In [ ]:
def build_subset(df: pd.DataFrame, keys: list, mapping: dict, df_name: str) -> pd.DataFrame:
    """공통 키 + 최종 변수명으로 축약된 DataFrame 생성 함수"""
    # 실제 컬럼명이 다르면 mapping 사전 안 후보 컬럼명을 수정하세요.
    if df.empty:
        return pd.DataFrame(columns=keys + list(mapping.keys()))

    result = pd.DataFrame()
    for key in keys:
        if key in df.columns:
            result[key] = df[key]
        else:
            result[key] = np.nan

    for final_col, candidates in mapping.items():
        found_col = next((col for col in candidates if col in df.columns), None)
        if found_col is not None:
            result[final_col] = df[found_col]
        else:
            result[final_col] = np.nan
            print(f'[CHECK] {df_name}: {final_col}에 대응되는 실제 컬럼명을 확인하세요 -> 후보 {candidates}')

    return result

# 실제 컬럼명 확인 후 수정
geunsu_store_map = {
    'store_cnt': ['store_cnt', '점포수', '가맹점수'],
    'franchise_cnt': ['franchise_cnt', '프랜차이즈수', 'franchise_store_cnt'],
}
geunsu_sales_map = {
    'sales_amt': ['sales_amt', '매출금액', 'sales'],
}
geunsu_land_map = {
    'avg_land_price': ['avg_land_price', '평균공시지가', 'land_price'],
    'land_price_growth': ['land_price_growth', '공시지가상승률', 'land_growth'],
}

jiryun_t13_map = {
    'move_cnt': ['move_cnt', 'CNT', 'cnt'],
    'sex_group': ['sex_group', 'SEX_CD', 'sex_cd'],
    'age_group': ['age_group', 'AGE_GRP', 'age_grp'],
}
jiryun_t25_map = {
    'stay_time': ['stay_time', 'DURATION', 'duration'],
}
jiryun_t26_map = {
    'move_purpose': ['move_purpose', 'PURPOSE', 'purpose'],
}
jiryun_t27_map = {
    'transport_type': ['transport_type', 'TRANS_GB', 'trans_gb'],
}

eunbi_deal_map = {
    'deal_cnt': ['deal_cnt', '거래건수', 'transaction_cnt'],
    'deal_growth': ['deal_growth', '거래증감률', 'transaction_cnt_growth'],
}
eunbi_pop_map = {
    'total_pop': ['total_pop', '총인구', 'population_total'],
    'working_age_pop': ['working_age_pop', '생산연령인구', 'working_pop'],
}
eunbi_firm_map = {
    'new_firm_cnt': ['new_firm_cnt', '신규기업수', 'new_company_cnt'],
}

seongju_credit_map = {
    'econ_activity_cnt': ['econ_activity_cnt', '경제활동건수', 'activity_cnt'],
    'credit_spending_proxy': ['credit_spending_proxy', '신용지출대리지표', 'credit_amt'],
}
seongju_bus_map = {
    'bus_access': ['bus_access', '버스접근성', 'bus_access_score'],
}
seongju_subway_map = {
    'subway_access': ['subway_access', '지하철접근성', 'subway_access_score'],
}
seongju_traffic_map = {
    'traffic_access_score': ['traffic_access_score', '교통접근성종합', 'traffic_score'],
}

sub_geunsu_store = build_subset(df_geunsu_store, selected_keys, geunsu_store_map, 'df_geunsu_store')
sub_geunsu_sales = build_subset(df_geunsu_sales, selected_keys, geunsu_sales_map, 'df_geunsu_sales')
sub_geunsu_landprice = build_subset(df_geunsu_landprice, selected_keys, geunsu_land_map, 'df_geunsu_landprice')

sub_jiryun_t13 = build_subset(df_jiryun_t13, selected_keys, jiryun_t13_map, 'df_jiryun_t13')
sub_jiryun_t25 = build_subset(df_jiryun_t25, selected_keys, jiryun_t25_map, 'df_jiryun_t25')
sub_jiryun_t26 = build_subset(df_jiryun_t26, selected_keys, jiryun_t26_map, 'df_jiryun_t26')
sub_jiryun_t27 = build_subset(df_jiryun_t27, selected_keys, jiryun_t27_map, 'df_jiryun_t27')

sub_eunbi_deal = build_subset(df_eunbi_deal, selected_keys, eunbi_deal_map, 'df_eunbi_deal')
sub_eunbi_pop = build_subset(df_eunbi_pop, selected_keys, eunbi_pop_map, 'df_eunbi_pop')
sub_eunbi_firm = build_subset(df_eunbi_firm, selected_keys, eunbi_firm_map, 'df_eunbi_firm')

sub_seongju_credit = build_subset(df_seongju_credit, selected_keys, seongju_credit_map, 'df_seongju_credit')
sub_seongju_bus = build_subset(df_seongju_bus, selected_keys, seongju_bus_map, 'df_seongju_bus')
sub_seongju_subway = build_subset(df_seongju_subway, selected_keys, seongju_subway_map, 'df_seongju_subway')
sub_seongju_traffic = build_subset(df_seongju_traffic, selected_keys, seongju_traffic_map, 'df_seongju_traffic')

## 8. 병합 전 점검

병합 전에는 각 축약 DataFrame이 정말 같은 키 구조를 갖고 있는지 확인해야 합니다. 특히 고유키 개수, 중복 키, 키 결측치가 중요합니다.

In [ ]:
subset_frames = {
    'sub_geunsu_store': sub_geunsu_store,
    'sub_geunsu_sales': sub_geunsu_sales,
    'sub_geunsu_landprice': sub_geunsu_landprice,
    'sub_jiryun_t13': sub_jiryun_t13,
    'sub_jiryun_t25': sub_jiryun_t25,
    'sub_jiryun_t26': sub_jiryun_t26,
    'sub_jiryun_t27': sub_jiryun_t27,
    'sub_eunbi_deal': sub_eunbi_deal,
    'sub_eunbi_pop': sub_eunbi_pop,
    'sub_eunbi_firm': sub_eunbi_firm,
    'sub_seongju_credit': sub_seongju_credit,
    'sub_seongju_bus': sub_seongju_bus,
    'sub_seongju_subway': sub_seongju_subway,
    'sub_seongju_traffic': sub_seongju_traffic,
}

def premerge_check(df: pd.DataFrame, name: str, keys: list):
    print(f'\n===== {name} =====')
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    if set(keys).issubset(df.columns):
        print('고유키 개수:', df[keys].drop_duplicates().shape[0])
        print('중복 키 개수:', df.duplicated(subset=keys).sum())
        print('키 결측치 개수:', df[keys].isna().sum().to_dict())
    else:
        print('[CHECK] 병합 키 누락:', {k: (k in df.columns) for k in keys})

for name, df in subset_frames.items():
    premerge_check(df, name, selected_keys)

print('\n[병합 전 표준화할 컬럼명 예시]')
print('- base_ym -> 기준년월')
print('- dong_cd -> 행정동코드')
print('- dong_name -> 행정동명')

## 9. master table 생성

master table은 최종 분석의 기준 테이블입니다. 일반적으로 `기준년월 + 행정동코드` 또는 `기준년월 + 블록코드` 단위가 됩니다. 아래 셀은 left join 기반으로 개인별 축약 데이터를 순서대로 붙입니다.

In [ ]:
def safe_merge(left: pd.DataFrame, right: pd.DataFrame, keys: list, tag: str) -> pd.DataFrame:
    if left.empty:
        print(f'[INFO] {tag}: left가 비어 있어 right를 기준으로 시작합니다.')
        return right.copy()
    if right.empty:
        print(f'[INFO] {tag}: right가 비어 있어 left를 그대로 유지합니다.')
        return left.copy()

    merged = left.merge(right, on=keys, how='left', suffixes=('', f'_{tag}'))
    print(f'[MERGE] {tag}: {left.shape} + {right.shape} -> {merged.shape}')
    return merged

# 1) 근수 파트 통합
geunsu_master = safe_merge(sub_geunsu_store, sub_geunsu_sales, selected_keys, 'geunsu_sales')
geunsu_master = safe_merge(geunsu_master, sub_geunsu_landprice, selected_keys, 'geunsu_land')

# 2) 지륜 파트 통합
jiryun_master = safe_merge(sub_jiryun_t13, sub_jiryun_t25, selected_keys, 'jiryun_t25')
jiryun_master = safe_merge(jiryun_master, sub_jiryun_t26, selected_keys, 'jiryun_t26')
jiryun_master = safe_merge(jiryun_master, sub_jiryun_t27, selected_keys, 'jiryun_t27')

# 3) 은비 파트 통합
eunbi_master = safe_merge(sub_eunbi_deal, sub_eunbi_pop, selected_keys, 'eunbi_pop')
eunbi_master = safe_merge(eunbi_master, sub_eunbi_firm, selected_keys, 'eunbi_firm')

# 4) 성주 파트 통합
seongju_master = safe_merge(sub_seongju_credit, sub_seongju_bus, selected_keys, 'seongju_bus')
seongju_master = safe_merge(seongju_master, sub_seongju_subway, selected_keys, 'seongju_subway')
seongju_master = safe_merge(seongju_master, sub_seongju_traffic, selected_keys, 'seongju_traffic')

# 전체 병합 시작 기준은 근수 파트로 두되, 비어 있으면 다음 파트로 넘어갑니다.
master_df = geunsu_master.copy()
if master_df.empty:
    master_df = jiryun_master.copy()
if master_df.empty:
    master_df = eunbi_master.copy()
if master_df.empty:
    master_df = seongju_master.copy()

master_df = safe_merge(master_df, jiryun_master, selected_keys, 'jiryun_master')
master_df = safe_merge(master_df, eunbi_master, selected_keys, 'eunbi_master')
master_df = safe_merge(master_df, seongju_master, selected_keys, 'seongju_master')

# 병합 후 완전히 중복된 컬럼명 제거
master_df = master_df.loc[:, ~master_df.columns.duplicated()].copy()

print('\nmaster_df shape:', master_df.shape)
display(master_df.head())

if not master_df.empty:
    master_df.to_csv(MASTER_TABLE_FILE, index=False, encoding='utf-8-sig')
    print('[SAVE] master table ->', MASTER_TABLE_FILE)

### master table 단위 설명 템플릿
- 이 master table의 단위: ________________________________________________
- 병합에 사용한 최종 키: ________________________________________________
- 한 행이 의미하는 바: 예) 특정 기준년월의 특정 행정동에 대한 통합 특성값 1개

## 10. master table 기본 점검

행 수, 열 수, 타입, 결측치, 중복 여부를 먼저 확인하면 이후 분석 오류를 줄일 수 있습니다.

In [ ]:
print('행 수 / 열 수:', master_df.shape)
print('\n[info()]')
master_df.info()

if not master_df.empty:
    print('\n[describe()]')
    display(master_df.describe(include='all').T.head(50))

    missing_cnt = master_df.isna().sum().sort_values(ascending=False)
    missing_ratio = (master_df.isna().mean() * 100).sort_values(ascending=False)
    missing_summary = pd.concat([missing_cnt.rename('missing_cnt'), missing_ratio.rename('missing_ratio_pct')], axis=1)
    print('\n[결측치 개수 및 비율 상위 30개]')
    display(missing_summary.head(30))

    print('\n[중복행 수]')
    print(master_df.duplicated().sum())

    obj_cols = master_df.select_dtypes(include=['object', 'category']).columns.tolist()
    if obj_cols:
        print('\n[범주형 값 확인]')
        for col in obj_cols[:10]:
            print(f'- {col}: {master_df[col].astype(str).value_counts(dropna=False).head(5).to_dict()}')

    num_cols = master_df.select_dtypes(include=np.number).columns.tolist()
    if num_cols:
        print('\n[주요 숫자형 컬럼 분포 확인용 요약]')
        display(master_df[num_cols].agg(['min', 'median', 'max']).T.head(30))

## 11. 최종 변수 연결표

아래 표는 최종 분석 변수 사전 역할을 합니다. 팀원이 이 표만 봐도 변수 의미와 사용 방향을 이해할 수 있어야 합니다.

| 데이터셋 | 원본 컬럼 | 최종 변수명 | 의미 | 사용 방향 |
|---|---|---|---|---|
| 거래량 | 거래건수 | deal_cnt | 거래 활성도 | 투자 유입 압력 |
| 거래량 | 증감률 | deal_growth | 거래 증가 속도 | 초기 변화 신호 |
| 공시지가 | 평균 공시지가 | avg_land_price | 지역 자산가치 수준 | 부동산 가치 수준 |
| 공시지가 | 상승률 | land_price_growth | 가격 상승 압력 | 젠트리피케이션 압력 |
| 인구 | 총인구 | total_pop | 배후수요 규모 | 소비 기반 |
| 인구 | 생산연령인구 | working_age_pop | 경제활동 중심층 | 상권 유지력 |
| 기업 | 신규기업수 | new_firm_cnt | 새 사업체 유입 | 상권 변화/진입 신호 |
| 통신 | CNT | move_cnt | 이동량/체류량 규모 | 상권 유입·활동량 |
| 통신 | DURATION | stay_time | 평균 체류시간 | 체류형 상권 여부 |
| 통신 | PURPOSE | move_purpose | 이동 목적 | 생활권/업무권 구분 |
| 통신 | TRANS_GB | transport_type | 이동수단 | 접근성/교통 특성 |
| 통신 | SEX_CD | sex_group | 성별 구성 | 이용자 특성 |
| 통신 | AGE_GRP | age_group | 연령대 구성 | 소비층 특성 |

## 12. EDA

EDA에서는 변수 분포, 이상치, 상관관계, 지역별 차이, 월별 추이를 확인합니다. 아래 코드는 matplotlib만 사용하도록 작성했습니다.

### 결과 해석 예문
- 무엇을 확인했는지: 핵심 수치형 변수의 분포, 월별 변화, 지역별 상·하위 차이를 확인했다.
- 어떤 패턴이 보였는지: 특정 지역에서 거래량과 공시지가 상승률이 동시에 높게 나타났고, 일부 변수는 오른쪽 꼬리가 긴 분포를 보였다.

In [ ]:
numeric_cols = master_df.select_dtypes(include=np.number).columns.tolist()
eda_cols = numeric_cols[:6]  # 너무 많으면 앞 6개만 먼저 확인

if eda_cols:
    # 1) 히스토그램
    master_df[eda_cols].hist(figsize=(14, 8), bins=30)
    plt.suptitle('Numeric Variable Histograms')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'eda_histograms.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 2) 박스플롯
    plt.figure(figsize=(12, 6))
    master_df[eda_cols].boxplot(rot=45)
    plt.title('Numeric Variable Boxplots')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'eda_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 3) 상관행렬
    corr = master_df[eda_cols].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect='auto')
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
    plt.yticks(range(len(corr.index)), corr.index)
    plt.colorbar()
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'eda_corr_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 4) 지역별 top / bottom 비교
    region_col = PRIMARY_REGION_NAME_COL if PRIMARY_REGION_NAME_COL in master_df.columns else None
    focus_col = eda_cols[0]
    if region_col is not None:
        region_summary = master_df.groupby(region_col, dropna=False)[focus_col].mean().sort_values()
        display(region_summary.head(5).rename('bottom_mean'))
        display(region_summary.tail(5).rename('top_mean'))

    # 5) 월별 추이 그래프
    if PRIMARY_DATE_COL in master_df.columns:
        monthly = master_df.groupby(PRIMARY_DATE_COL, dropna=False)[eda_cols].mean(numeric_only=True)
        monthly.plot(figsize=(12, 6))
        plt.title('Monthly Trend of Key Variables')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'eda_monthly_trend.png', dpi=150, bbox_inches='tight')
        plt.show()

    # 6) 위험 후보 변수 분포
    if RISK_PROXY_COL in master_df.columns:
        plt.figure(figsize=(8, 4))
        master_df[RISK_PROXY_COL].dropna().hist(bins=30)
        plt.title(f'Distribution of {RISK_PROXY_COL}')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'eda_risk_proxy_distribution.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('[INFO] EDA에 사용할 숫자형 컬럼이 없습니다.')

## 13. 통계 검정

### 통계 검정 목적
통계 검정은 변수 간 차이와 관계가 우연인지, 일정한 패턴이 있는지 확인하기 위한 단계입니다.

### 초보자용 해석 안내
- 일반적으로 p-value가 0.05보다 작으면 통계적으로 유의하다고 해석하는 경우가 많습니다.
- 하지만 표본 수, 데이터 품질, 변수 의미를 함께 보고 판단해야 합니다.

In [ ]:
stats_rows = []

group_col_for_stats = None
if TARGET_COL in master_df.columns:
    group_col_for_stats = TARGET_COL
elif RISK_PROXY_COL in master_df.columns and master_df[RISK_PROXY_COL].notna().sum() > 0:
    group_col_for_stats = '__risk_proxy_group__'
    median_value = master_df[RISK_PROXY_COL].median()
    master_df[group_col_for_stats] = np.where(master_df[RISK_PROXY_COL] >= median_value, 'high', 'low')

num_cols = master_df.select_dtypes(include=np.number).columns.tolist()
cat_cols = master_df.select_dtypes(include=['object', 'category']).columns.tolist()

# 1) t-test / Mann-Whitney U
if group_col_for_stats is not None and group_col_for_stats in master_df.columns:
    group_values = master_df[group_col_for_stats].dropna().unique().tolist()
    if len(group_values) >= 2:
        g1, g2 = group_values[:2]
        for col in num_cols[:10]:
            temp = master_df[[group_col_for_stats, col]].dropna()
            if temp[group_col_for_stats].nunique() < 2:
                continue
            x = temp.loc[temp[group_col_for_stats] == g1, col]
            y = temp.loc[temp[group_col_for_stats] == g2, col]
            if len(x) > 1 and len(y) > 1:
                t_stat, t_p = stats.ttest_ind(x, y, equal_var=False, nan_policy='omit')
                mw_stat, mw_p = stats.mannwhitneyu(x, y, alternative='two-sided')
                stats_rows.append({'test': 't-test', 'variable': col, 'group_col': group_col_for_stats, 'statistic': t_stat, 'p_value': t_p})
                stats_rows.append({'test': 'Mann-Whitney U', 'variable': col, 'group_col': group_col_for_stats, 'statistic': mw_stat, 'p_value': mw_p})

# 2) 카이제곱
if group_col_for_stats is not None and group_col_for_stats in master_df.columns:
    for col in cat_cols[:10]:
        if col == group_col_for_stats:
            continue
        temp = master_df[[group_col_for_stats, col]].dropna()
        if temp.empty or temp[group_col_for_stats].nunique() < 2 or temp[col].nunique() < 2:
            continue
        contingency = pd.crosstab(temp[group_col_for_stats], temp[col])
        chi2, p_val, dof, _ = stats.chi2_contingency(contingency)
        stats_rows.append({'test': 'Chi-square', 'variable': col, 'group_col': group_col_for_stats, 'statistic': chi2, 'p_value': p_val})

# 3) Pearson / Spearman 상관
if len(num_cols) >= 2:
    base_col = num_cols[0]
    for col in num_cols[1:10]:
        temp = master_df[[base_col, col]].dropna()
        if len(temp) > 2:
            pearson_r, pearson_p = stats.pearsonr(temp[base_col], temp[col])
            spearman_r, spearman_p = stats.spearmanr(temp[base_col], temp[col])
            stats_rows.append({'test': f'Pearson ({base_col})', 'variable': col, 'group_col': '', 'statistic': pearson_r, 'p_value': pearson_p})
            stats_rows.append({'test': f'Spearman ({base_col})', 'variable': col, 'group_col': '', 'statistic': spearman_r, 'p_value': spearman_p})

stats_result_df = pd.DataFrame(stats_rows)
if not stats_result_df.empty:
    stats_result_df['is_significant_0_05'] = stats_result_df['p_value'] < 0.05
    display(stats_result_df.head(30))
    stats_result_df.to_csv(STATS_RESULT_FILE, index=False, encoding='utf-8-sig')
    print('[SAVE] stats result ->', STATS_RESULT_FILE)
else:
    print('[INFO] 통계 검정에 사용할 데이터가 충분하지 않습니다.')

## 14. 머신러닝 준비

모델링 전에는 target 변수와 feature 목록을 먼저 정해야 합니다. 현재 target이 없다면 위험 점수 proxy를 활용해 임시 target을 만들 수도 있습니다.

### target 예시
- 위험 단계: `risk_label`
- 위험 점수: `risk_score`
- 군집 라벨: `cluster`

### 초보자 안내
- 지도학습: 정답(target)이 있을 때 사용
- 비지도학습: 정답이 없고 패턴이나 군집을 찾고 싶을 때 사용

In [ ]:
model_df = master_df.copy()

# target 컬럼이 실제로 없으면 아래 예시처럼 proxy 기준으로 임시 target을 만듭니다.
# 실제 프로젝트에서는 팀이 합의한 target 정의로 반드시 교체하세요.
if TARGET_COL not in model_df.columns:
    if RISK_PROXY_COL in model_df.columns and model_df[RISK_PROXY_COL].notna().sum() > 0:
        threshold = model_df[RISK_PROXY_COL].quantile(0.75)
        model_df[TARGET_COL] = np.where(model_df[RISK_PROXY_COL] >= threshold, 1, 0)
        print(f'[INFO] {TARGET_COL}이 없어 {RISK_PROXY_COL} 기준 상위 25%를 1로 설정했습니다.')
    else:
        model_df[TARGET_COL] = np.nan
        print(f'[CHECK] {TARGET_COL}과 {RISK_PROXY_COL} 모두 확인 필요')

exclude_cols = selected_keys + [PRIMARY_REGION_NAME_COL, TARGET_COL]
feature_cols = [col for col in model_df.columns if col not in exclude_cols]
feature_cols = [col for col in feature_cols if model_df[col].notna().sum() > 0]

print('target column :', TARGET_COL)
print('feature count :', len(feature_cols))
print('sample features:', feature_cols[:20])

ml_ready_df = model_df[selected_keys + [TARGET_COL] + feature_cols].copy()
ml_ready_df = ml_ready_df.dropna(subset=[TARGET_COL])

X = ml_ready_df[feature_cols].copy() if feature_cols else pd.DataFrame()
y = ml_ready_df[TARGET_COL].copy() if TARGET_COL in ml_ready_df.columns else pd.Series(dtype=float)

numeric_features = X.select_dtypes(include=np.number).columns.tolist() if not X.empty else []
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist() if not X.empty else []

print('numeric feature count    :', len(numeric_features))
print('categorical feature count:', len(categorical_features))

# 결측치 처리 / 인코딩 / 스케일링 설명
# - 숫자형: 중앙값으로 결측치 대체, 필요 시 스케일링
# - 범주형: 최빈값 대체 후 OneHotEncoder 적용
# - 트리 모델은 스케일링 영향이 상대적으로 적고, 선형 모델은 스케일링이 더 중요할 수 있습니다.

preprocessor_for_tree = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ],
    remainder='drop'
)

preprocessor_for_linear = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ],
    remainder='drop'
)

if not X.empty and len(y) > 10:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y if y.nunique() <= 10 else None
    )
    print('train shape:', X_train.shape, 'test shape:', X_test.shape)
else:
    X_train, X_test, y_train, y_test = pd.DataFrame(), pd.DataFrame(), pd.Series(dtype=float), pd.Series(dtype=float)
    print('[INFO] 머신러닝용 데이터가 충분하지 않습니다.')

## 15. 머신러닝 모델 1

### 모델 1 이름
RandomForestClassifier 또는 RandomForestRegressor

### 왜 사용하는가?
트리 기반 모델은 변수 간 비선형 관계를 비교적 잘 포착하고, 변수 중요도를 확인하기 쉬워 초반 기준 모델로 자주 사용합니다.

### 결과 해석 방법
- 분류 문제: 정확도, F1-score 확인
- 회귀 문제: MAE, RMSE, R² 확인
- 변수 중요도: 어떤 변수가 모델 판단에 크게 기여했는지 확인

In [ ]:
model_results = []
rf_feature_importance_df = pd.DataFrame()
rf_pipeline = None

if not X_train.empty and len(y_train) > 0:
    is_classification = y_train.nunique() <= 10

    if is_classification:
        rf_model = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight='balanced')
    else:
        rf_model = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)

    rf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_for_tree),
        ('model', rf_model),
    ])

    rf_pipeline.fit(X_train, y_train)
    rf_pred = rf_pipeline.predict(X_test)

    if is_classification:
        rf_acc = accuracy_score(y_test, rf_pred)
        rf_f1 = f1_score(y_test, rf_pred, average='weighted')
        model_results.append({'model': 'RandomForestClassifier', 'metric_1': 'accuracy', 'value_1': rf_acc, 'metric_2': 'f1_weighted', 'value_2': rf_f1})
        print(classification_report(y_test, rf_pred))
    else:
        rf_mae = mean_absolute_error(y_test, rf_pred)
        rf_rmse = mean_squared_error(y_test, rf_pred, squared=False)
        rf_r2 = r2_score(y_test, rf_pred)
        model_results.append({'model': 'RandomForestRegressor', 'metric_1': 'mae', 'value_1': rf_mae, 'metric_2': 'rmse', 'value_2': rf_rmse, 'metric_3': 'r2', 'value_3': rf_r2})

    # feature importance 출력
    try:
        feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
        feature_importance = rf_pipeline.named_steps['model'].feature_importances_
        rf_feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': feature_importance}).sort_values('importance', ascending=False)
        display(rf_feature_importance_df.head(20))
    except Exception as exc:
        print('[INFO] feature importance 추출 실패:', exc)
else:
    print('[INFO] 모델 1을 학습할 데이터가 충분하지 않습니다.')

## 16. 머신러닝 모델 2

### 모델 2 이름
GradientBoosting 또는 LogisticRegression / LinearRegression

### 모델 1과 비교 포인트
- 성능 지표 차이
- 해석 용이성
- 데이터 크기와 결측치 대응 편의성

In [ ]:
model2_pipeline = None
model2_feature_importance_df = pd.DataFrame()

if not X_train.empty and len(y_train) > 0:
    is_classification = y_train.nunique() <= 10

    if is_classification:
        # XGBoost가 없을 때 대체 가능한 기본 예시
        model2 = GradientBoostingClassifier(random_state=RANDOM_STATE)
        model2_name = 'GradientBoostingClassifier'
        preprocessor = preprocessor_for_tree
    else:
        model2 = GradientBoostingRegressor(random_state=RANDOM_STATE)
        model2_name = 'GradientBoostingRegressor'
        preprocessor = preprocessor_for_tree

    # 선형모델로 바꾸고 싶으면 아래 주석을 참고하세요.
    # if is_classification:
    #     model2 = LogisticRegression(max_iter=1000)
    #     model2_name = 'LogisticRegression'
    #     preprocessor = preprocessor_for_linear
    # else:
    #     model2 = LinearRegression()
    #     model2_name = 'LinearRegression'
    #     preprocessor = preprocessor_for_linear

    model2_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model2)])
    model2_pipeline.fit(X_train, y_train)
    model2_pred = model2_pipeline.predict(X_test)

    if is_classification:
        m2_acc = accuracy_score(y_test, model2_pred)
        m2_f1 = f1_score(y_test, model2_pred, average='weighted')
        model_results.append({'model': model2_name, 'metric_1': 'accuracy', 'value_1': m2_acc, 'metric_2': 'f1_weighted', 'value_2': m2_f1})
    else:
        m2_mae = mean_absolute_error(y_test, model2_pred)
        m2_rmse = mean_squared_error(y_test, model2_pred, squared=False)
        m2_r2 = r2_score(y_test, model2_pred)
        model_results.append({'model': model2_name, 'metric_1': 'mae', 'value_1': m2_mae, 'metric_2': 'rmse', 'value_2': m2_rmse, 'metric_3': 'r2', 'value_3': m2_r2})

    try:
        feature_names = model2_pipeline.named_steps['preprocessor'].get_feature_names_out()
        if hasattr(model2_pipeline.named_steps['model'], 'feature_importances_'):
            importance_values = model2_pipeline.named_steps['model'].feature_importances_
            model2_feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance_values}).sort_values('importance', ascending=False)
            display(model2_feature_importance_df.head(20))
    except Exception as exc:
        print('[INFO] 모델 2 중요도 추출 실패:', exc)
else:
    print('[INFO] 모델 2를 학습할 데이터가 충분하지 않습니다.')

## 17. 비지도학습 대안

정답(target)이 충분히 정의되지 않았을 때는 KMeans 같은 군집화로 지역 유형을 먼저 나눠볼 수 있습니다.

In [ ]:
cluster_df = master_df.copy()
cluster_result_df = pd.DataFrame()

cluster_num_cols = cluster_df.select_dtypes(include=np.number).columns.tolist()
cluster_num_cols = [col for col in cluster_num_cols if cluster_df[col].notna().sum() > 0]

if len(cluster_num_cols) >= 2:
    temp_cluster = cluster_df[selected_keys + cluster_num_cols].copy()
    temp_cluster[cluster_num_cols] = temp_cluster[cluster_num_cols].fillna(temp_cluster[cluster_num_cols].median())

    kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
    temp_cluster['cluster'] = kmeans.fit_predict(temp_cluster[cluster_num_cols])
    cluster_result_df = temp_cluster.copy()

    print('[군집별 평균 특성]')
    display(cluster_result_df.groupby('cluster')[cluster_num_cols].mean())

    plt.figure(figsize=(8, 6))
    x_col, y_col = cluster_num_cols[:2]
    for cluster_value, temp in cluster_result_df.groupby('cluster'):
        plt.scatter(temp[x_col], temp[y_col], label=f'cluster {cluster_value}', alpha=0.7)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title('KMeans Cluster Example')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'kmeans_cluster_example.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('[INFO] KMeans에 사용할 숫자형 컬럼이 부족합니다.')

## 18. 모델 비교 및 해석

성능표와 중요 변수를 함께 보면서 어떤 모델을 최종 채택할지 결정합니다.

### 최종 채택 모델 적는 칸
- 최종 채택 모델: ________________________________________________
- 선택 이유 예문: 성능 지표가 가장 안정적이었고, 변수 중요도 해석이 가능하며 팀 설명에도 적합했기 때문에 선택했다.

In [ ]:
model_compare_df = pd.DataFrame(model_results)
display(model_compare_df)

if not model_compare_df.empty:
    model_compare_df.to_csv(MODEL_COMPARE_FILE, index=False, encoding='utf-8-sig')
    print('[SAVE] model compare ->', MODEL_COMPARE_FILE)

print('\n[RandomForest 중요 변수 상위]')
display(rf_feature_importance_df.head(10))

print('\n[Model2 중요 변수 상위]')
display(model2_feature_importance_df.head(10))

# SHAP 예시
# SHAP 패키지가 설치되어 있으면 모델 해석을 더 자세히 볼 수 있습니다.
# 설치가 안 되어 있으면 아래처럼 대체 설명을 남깁니다.
try:
    import shap
    print('[INFO] SHAP 사용 가능: 필요 시 rf_pipeline 또는 model2_pipeline에 대해 explainer를 적용하세요.')
except Exception:
    print('[INFO] SHAP이 설치되어 있지 않습니다. 현재는 feature importance와 계수/중요도 비교로 해석합니다.')

## 19. 위험도 점수 및 위험단계 생성

위험도 점수는 최종적으로 대시보드와 정책 해석에 가장 직접적으로 활용되는 값입니다. 아래 예시는 표준화된 주요 변수 평균을 이용한 단순 템플릿입니다. 실제 프로젝트에서는 가중치나 모델 예측값으로 교체할 수 있습니다.

In [ ]:
risk_df = master_df.copy()

risk_candidate_cols = [
    col for col in ['deal_growth', 'land_price_growth', 'new_firm_cnt', 'move_cnt', 'traffic_access_score', 'credit_spending_proxy']
    if col in risk_df.columns
]

if risk_candidate_cols:
    temp = risk_df[risk_candidate_cols].copy()
    temp = temp.apply(pd.to_numeric, errors='coerce')
    temp = temp.fillna(temp.median())

    z_temp = (temp - temp.mean()) / temp.std(ddof=0).replace(0, np.nan)
    z_temp = z_temp.fillna(0)

    risk_df['risk_score'] = z_temp.mean(axis=1)
    risk_df['risk_score_rank'] = risk_df['risk_score'].rank(method='average', pct=True)

    risk_df['risk_stage'] = pd.cut(
        risk_df['risk_score_rank'],
        bins=[-0.01, 0.25, 0.5, 0.75, 1.0],
        labels=['저위험', '주의', '경계', '위험']
    )

    region_cols_for_summary = [col for col in [PRIMARY_REGION_CODE_COL, PRIMARY_REGION_NAME_COL] if col in risk_df.columns]
    risk_summary_df = risk_df.groupby(region_cols_for_summary, dropna=False).agg(
        avg_risk_score=('risk_score', 'mean'),
        max_risk_stage=('risk_stage', 'max')
    ).reset_index() if region_cols_for_summary else pd.DataFrame()

    display(risk_df.head())
    display(risk_summary_df.head())
else:
    risk_df['risk_score'] = np.nan
    risk_df['risk_stage'] = np.nan
    risk_summary_df = pd.DataFrame()
    print('[INFO] 위험 점수를 만들 핵심 후보 변수가 부족합니다.')

## 20. 대시보드용 export 파일 생성

대시보드용 파일은 시각화 도구에서 바로 읽기 쉽도록 최소한의 핵심 컬럼만 포함하는 것이 좋습니다.

### 태블로 / 피그마 / 대시보드 활용 설명 템플릿
- 이 파일은 지역별 위험 단계와 핵심 지표를 한 번에 보여주는 테이블로 활용한다.
- 태블로에서는 월별 필터, 지역 필터, 위험 단계 색상 구분에 활용할 수 있다.
- 피그마 시안에서는 카드형 요약 수치, 지도형 위험도, 추이 그래프 데이터 소스로 사용할 수 있다.

In [ ]:
export_df = risk_df.copy()

dashboard_cols = [
    PRIMARY_DATE_COL,
    PRIMARY_REGION_CODE_COL,
    PRIMARY_REGION_NAME_COL,
    'store_cnt', 'franchise_cnt', 'sales_amt', 'avg_land_price', 'land_price_growth',
    'deal_cnt', 'deal_growth', 'total_pop', 'working_age_pop', 'new_firm_cnt',
    'move_cnt', 'stay_time', 'move_purpose', 'transport_type',
    'econ_activity_cnt', 'credit_spending_proxy', 'bus_access', 'subway_access', 'traffic_access_score',
    'risk_score', 'risk_stage'
]
dashboard_cols = [col for col in dashboard_cols if col in export_df.columns]
export_df = export_df[dashboard_cols].copy()

if not export_df.empty:
    export_df.to_csv(DASHBOARD_EXPORT_FILE, index=False, encoding='utf-8-sig')
    print('[SAVE] dashboard export ->', DASHBOARD_EXPORT_FILE)
    display(export_df.head())
else:
    print('[INFO] export_df가 비어 있습니다.')

summary_text = '''# 공모전 제출용 핵심 결과 정리

1) 프로젝트 목적
- 여기에 작성

2) 사용 데이터
- 여기에 작성

3) 핵심 변수
- 여기에 작성

4) 통계 검정 요약
- 여기에 작성

5) 머신러닝 결과 요약
- 여기에 작성

6) 정책적 시사점
- 여기에 작성

7) 대시보드 활용 방안
- 여기에 작성
'''
FINAL_SUMMARY_FILE.write_text(summary_text, encoding='utf-8')
print('[SAVE] summary note ->', FINAL_SUMMARY_FILE)

## 21. 최종 산출물 목록

최종 파일명과 용도를 한 번에 정리합니다.

| 산출물 | 파일명 | 설명 |
|---|---|---|
| master table | master_table.csv | 개인별 결과 통합본 |
| stats result | stats_result.csv | 통계 검정 결과 |
| ml result | model_compare.csv | 모델 비교 결과 |
| dashboard export | dashboard_export.csv | 대시보드용 결과 |
| summary note | final_summary.md | 공모전 제출용 핵심 정리 |

## 22. 공모전 제출용 핵심 결과 정리

### 아래 항목을 그대로 작성하면 됩니다
1) 프로젝트 목적
- 성남시 내 지역별 젠트리피케이션 위험도를 조기 진단하고, 상생 정책 설계를 지원하기 위해 통합 분석 시스템을 구축했다.

2) 사용 데이터
- 가맹점, 매출, 공시지가, 통신, 거래량, 인구, 기업, 신용, 교통 접근성 데이터를 사용했다.

3) 핵심 변수
- 거래량, 공시지가 상승률, 인구 구조, 기업 유입, 통신 이동량, 경제활동/교통 접근성 변수를 핵심 변수로 사용했다.

4) 통계 검정 요약
- 위험 고/저 그룹 간 차이와 핵심 변수 간 상관관계를 검정해 주요 패턴을 확인했다.

5) 머신러닝 결과 요약
- 두 개 이상의 모델을 비교해 위험도 예측 또는 분류에 적합한 모델을 선정했다.

6) 정책적 시사점
- 위험도가 높은 지역은 임대료 부담, 상권 변화, 접근성 개선 효과 등을 함께 고려한 선제적 정책이 필요하다.

7) 대시보드 활용 방안
- 월별 위험 추이, 지역별 비교, 핵심 변수 drill-down을 제공하는 시각화 대시보드로 활용한다.

## 23. 최종 체크리스트

- [ ] 개인별 clean 파일 모두 불러오기 완료
- [ ] 공통 키 통일 완료
- [ ] master table 생성 완료
- [ ] EDA 완료
- [ ] 통계 검정 완료
- [ ] 모델 2개 적용 완료
- [ ] 모델 비교 완료
- [ ] 위험도 점수 생성 완료
- [ ] 대시보드 export 완료
- [ ] 공모전 요약문 작성 완료

## 24. 팀 공유용 5줄 요약

### 슬랙 / 보고용 바로 복붙 템플릿
1. 개인별 전처리 결과를 공통 키 기준으로 통합해 master table을 만들었습니다.
2. 핵심 변수의 분포와 지역별·월별 패턴을 EDA 및 통계 검정으로 확인했습니다.
3. 머신러닝 모델 2개를 비교해 위험도 예측에 적합한 기준 모델을 검토했습니다.
4. 최종적으로 위험도 점수와 위험 단계를 생성해 지역별 위험 수준을 요약했습니다.
5. 대시보드용 export 파일과 공모전 제출용 요약 문안을 함께 정리했습니다.